[<img src="https://colab.research.google.com/assets/colab-badge.svg" align="left" alt="Open In Colab"/>](https://colab.research.google.com/github/luiscuellarh/apigee-unified-ai-gateway/blob/main/03_apigee_mcp_gateway.ipynb)

<br><br>


# Apigee AI Gateway — MCP Gateway Governance & Policies (AFT)

**Repository Location:** [`https://github.com/luiscuellarh/apigee-unified-ai-gateway/blob/main/03_apigee_mcp_gateway.ipynb`](https://github.com/luiscuellarh/apigee-unified-ai-gateway/blob/main/03_apigee_mcp_gateway.ipynb)  
**Execution Constraint:** This notebook is version-controlled in GitHub and designed to run cleanly when opened directly from that repository path in Google Colab (no local-only paths, no machine-specific assumptions, and no state carried between sessions).

**Framework & Toolkit:**
- **Toolkit:** **AFT (`aft` — Apigee Foundation Toolkit / Apigee Feature Templater)** (`https://github.com/apigee/apigee-templater`)
- **Repository:** [`https://github.com/luiscuellarh/apigee-unified-ai-gateway`](https://github.com/luiscuellarh/apigee-unified-ai-gateway)
- **Assumptions:** Apigee X is already provisioned with `APIGEE_ENV` attached to an environment group (`ENV_GROUP`), and target MCP backend services are reachable.

---

## Step 0 — Preflight (Single-Run Blocking Gate)
Run **Step 0** once to execute all four preflight stages (`0.1`–`0.4`) in a single execution:
- **0.1** Inputs & Global Configuration
- **0.2** Verify Apigee Topology, APIs & IAM
- **0.3** Verify Service Account Roles, Permissions & `<GoogleAccessToken>` Bindings + Live Gemini
- **0.4** Install AFT CLI & Preflight Summary Gate


In [ ]:
#@title 0.1–0.4 Step 0 Preflight (Inputs, Apigee Topology & IAM, Service Account & Live Gemini, AFT CLI & Summary Gate)
# Runs 0.1, 0.2, 0.3, and 0.4 sequentially in a single execution.
import json, os, pathlib, re, shutil, subprocess, time, urllib.error, urllib.request
import requests

# ============================================================================
# 0.1 Inputs & Global Configuration
# ============================================================================
print("▶ [0.1] Inputs & Global Configuration...")
try:
    from google.colab import auth
    auth.authenticate_user()
    print("✅ Authenticated successfully with Google Cloud in Colab.")
except ImportError:
    print("ℹ️ Running in local/standard Jupyter environment.")

#@markdown ### Project & Apigee
PROJECT_ID = "" #@param {type:"string"}
APIGEE_ORG = "" #@param {type:"string"}
APIGEE_ENV = "dev" #@param {type:"string"}
ENV_GROUP = "dev-group" #@param {type:"string"}
REGION = "europe-west1" #@param {type:"string"}
HOSTNAME = "" #@param {type:"string"}

#@markdown ### Service Account
SERVICE_ACCOUNT = "" #@param {type:"string"}

#@markdown ### Gemini
GEMINI_MODEL = "gemini-2.5-flash" #@param {type:"string"}

#@markdown ### Behaviour Flags
ENABLE_APIS = False #@param {type:"boolean"}

PROJECT_ID, APIGEE_ENV, ENV_GROUP, REGION, HOSTNAME, SERVICE_ACCOUNT = (
    v.strip() for v in (PROJECT_ID, APIGEE_ENV, ENV_GROUP, REGION, HOSTNAME, SERVICE_ACCOUNT)
)
APIGEE_ORG = APIGEE_ORG.strip() or PROJECT_ID
APIGEE_ENVIRONMENT = APIGEE_ENV
GEMINI_MODEL = GEMINI_MODEL.strip().removeprefix("google/")

missing = [k for k, v in {"PROJECT_ID": PROJECT_ID, "APIGEE_ENV": APIGEE_ENV, "REGION": REGION, "GEMINI_MODEL": GEMINI_MODEL}.items() if not v]
if missing:
    raise SystemExit(f"❌ HALT: Missing required parameter(s): {", ".join(missing)}.")

# If SERVICE_ACCOUNT is provided, do not ask for a name and do not create a new one
SA_PROVIDED = bool(SERVICE_ACCOUNT)
if SERVICE_ACCOUNT and "@" not in SERVICE_ACCOUNT:
    SERVICE_ACCOUNT = f"{SERVICE_ACCOUNT}@{PROJECT_ID}.iam.gserviceaccount.com"
if SERVICE_ACCOUNT and not re.fullmatch(r"[a-zA-Z0-9._-]+@[a-zA-Z0-9.-]+\.gserviceaccount\.com", SERVICE_ACCOUNT):
    raise SystemExit(f"❌ HALT: Invalid SERVICE_ACCOUNT '{SERVICE_ACCOUNT}' (expected *.gserviceaccount.com).")

os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "PROJECT_ID": PROJECT_ID,
    "APIGEE_ORG": APIGEE_ORG,
    "APIGEE_ENV": APIGEE_ENV,
    "APIGEE_ENVIRONMENT": APIGEE_ENVIRONMENT,
    "ENV_GROUP": ENV_GROUP,
    "REGION": REGION,
    "HOSTNAME": HOSTNAME,
    "SERVICE_ACCOUNT": SERVICE_ACCOUNT,
    "GEMINI_MODEL": GEMINI_MODEL,
})
subprocess.run(["gcloud", "config", "set", "project", PROJECT_ID, "--quiet"], check=True, capture_output=True)

PREFLIGHT = {"inputs": True, "apigee": False, "service_account": False, "gemini": False, "aft": False, "repo": False}
sa_mode = f"{SERVICE_ACCOUNT} (provided — verify only, no creation)" if SA_PROVIDED else f"apigee-ai-gateway-sa@{PROJECT_ID}.iam.gserviceaccount.com (auto-provision if missing)"
print(f"✅ [0.1] Inputs ready | project={PROJECT_ID} | org={APIGEE_ORG} | env={APIGEE_ENV} ({ENV_GROUP}) | region={REGION} | sa={sa_mode}")

# ============================================================================
# 0.2 Verify Apigee Topology, APIs & IAM
# ============================================================================
print("\n▶ [0.2] Verify Apigee Topology, APIs & IAM...")
def get_auth_token() -> str:
    return subprocess.check_output(["gcloud", "auth", "print-access-token"], text=True).strip()

def gcp_request(method: str, url: str, json_body=None, headers=None):
    req_headers = {"Authorization": f"Bearer {get_auth_token()}", "Content-Type": "application/json"}
    if headers:
        req_headers.update(headers)
    try:
        resp = requests.request(method, url, json=json_body, headers=req_headers, timeout=30)
        return resp.status_code, (resp.json() if resp.text else {})
    except ValueError:
        return resp.status_code, {"raw": resp.text}
    except Exception as exc:
        return 0, {"error": str(exc), "url": url}

def wait_for_proxy_ready(proxy_name: str, env_name: str = None, timeout_sec: int = 180) -> str:
    env_target = env_name or APIGEE_ENV
    list_url = f"{APIGEE_BASE}/environments/{env_target}/apis/{proxy_name}/deployments"
    deadline = time.time() + timeout_sec
    last_rev, last_state = None, "NOT_DEPLOYED"
    print(f"⏳ Waiting for proxy '{proxy_name}' to reach READY state in env '{env_target}'...")
    while time.time() < deadline:
        s_list, d_list = gcp_request("GET", list_url)
        deployments = d_list.get("deployments", []) if s_list == 200 else []
        if deployments:
            latest = sorted(deployments, key=lambda d: int(d.get("revision", 0)))[-1]
            last_rev = latest.get("revision")
            rev_url = f"{APIGEE_BASE}/environments/{env_target}/apis/{proxy_name}/revisions/{last_rev}/deployments"
            s_rev, d_rev = gcp_request("GET", rev_url)
            last_state = d_rev.get("state", latest.get("state", "UNKNOWN")) if s_rev == 200 else "UNKNOWN"
            if last_state == "READY":
                print(f"✅ Proxy '{proxy_name}' (revision {last_rev}) is READY in env '{env_target}'.")
                return str(last_rev)
            print(f"   • proxy={proxy_name} | rev={last_rev} | state={last_state} — waiting 5s...")
        else:
            print(f"   • proxy={proxy_name} | state=WAITING_FOR_DEPLOYMENT — waiting 5s...")
        time.sleep(5)
    raise SystemExit(
        f"❌ HALT: Timed out waiting for proxy '{proxy_name}' to become READY in env '{env_target}' "
        f"(last revision={last_rev}, state={last_state})."
    )

ACTIVE_IDENTITY = subprocess.check_output(["gcloud", "config", "get-value", "account"], text=True).strip()
if not ACTIVE_IDENTITY:
    _, t_info = gcp_request("GET", f"https://oauth2.googleapis.com/tokeninfo?access_token={get_auth_token()}")
    ACTIVE_IDENTITY = t_info.get("email", "")
if not ACTIVE_IDENTITY:
    raise SystemExit("❌ HALT: Could not resolve active caller identity.")
caller_member_prefix = "serviceAccount" if ACTIVE_IDENTITY.endswith(".gserviceaccount.com") else "user"
print(f"✅ Active identity: {ACTIVE_IDENTITY}")

# 1. Verify required GCP APIs
REQUIRED_SERVICES = ["apigee.googleapis.com", "cloudresourcemanager.googleapis.com", "iamcredentials.googleapis.com", "aiplatform.googleapis.com"]
disabled_svcs = [
    svc for svc in REQUIRED_SERVICES
    if gcp_request("GET", f"https://serviceusage.googleapis.com/v1/projects/{PROJECT_ID}/services/{svc}")[1].get("state") != "ENABLED"
]
if disabled_svcs:
    if ENABLE_APIS or disabled_svcs == ["iamcredentials.googleapis.com"]:
        gcp_request("POST", f"https://serviceusage.googleapis.com/v1/projects/{PROJECT_ID}/services:batchEnable", {"serviceIds": disabled_svcs})
        time.sleep(3)
    else:
        raise SystemExit(f"❌ HALT: Disabled API(s): {", ".join(disabled_svcs)}. Set ENABLE_APIS=True in Step 0 and re-run.")
print(f"✅ Required APIs ENABLED: {", ".join(REQUIRED_SERVICES)}")

# 2. Validate Apigee Organization, Environment, and Environment Group Hostname
APIGEE_BASE = f"https://apigee.googleapis.com/v1/organizations/{APIGEE_ORG}"
status, org_data = gcp_request("GET", APIGEE_BASE)
if status != 200 or org_data.get("state") != "ACTIVE":
    raise SystemExit(f"❌ HALT: Apigee org '{APIGEE_ORG}' not ACTIVE (HTTP {status}): {org_data}")

status, env_data = gcp_request("GET", f"{APIGEE_BASE}/environments/{APIGEE_ENV}")
if status != 200:
    raise SystemExit(f"❌ HALT: Apigee env '{APIGEE_ENV}' not found in org '{APIGEE_ORG}' (HTTP {status}): {env_data}")

_, eg_list = gcp_request("GET", f"{APIGEE_BASE}/envgroups")
matched_group, matched_hosts = None, []
for eg in eg_list.get("environmentGroups", []):
    eg_name = eg["name"]
    if ENV_GROUP and eg_name != ENV_GROUP:
        continue
    _, att_data = gcp_request("GET", f"{APIGEE_BASE}/envgroups/{eg_name}/attachments")
    if APIGEE_ENV in [a.get("environment") for a in att_data.get("environmentGroupAttachments", [])]:
        matched_group, matched_hosts = eg_name, eg.get("hostnames", [])
        break

if not matched_group or (not HOSTNAME and not matched_hosts):
    raise SystemExit(f"❌ HALT: Env '{APIGEE_ENV}' is not attached to a valid envgroup with a hostname in '{APIGEE_ORG}'.")
ENV_GROUP, HOSTNAME = matched_group, (HOSTNAME or matched_hosts[0])
os.environ.update({"ENV_GROUP": ENV_GROUP, "HOSTNAME": HOSTNAME})
print(f"✅ Apigee Topology: org={APIGEE_ORG} | env={APIGEE_ENV} | envgroup={ENV_GROUP} | host={HOSTNAME}")

# 3. Verify Effective Apigee IAM Permissions
REQUIRED_APIGEE_PERMS = [
    "apigee.proxies.create", "apigee.proxies.get", "apigee.proxies.update", "apigee.proxies.delete",
    "apigee.proxyrevisions.get", "apigee.proxyrevisions.update", "apigee.proxyrevisions.delete",
    "apigee.proxyrevisions.deploy", "apigee.proxyrevisions.undeploy",
    "apigee.deployments.create", "apigee.deployments.get", "apigee.deployments.update", "apigee.deployments.delete",
    "apigee.sharedflows.create", "apigee.sharedflows.get", "apigee.sharedflows.delete",
    "apigee.sharedflowrevisions.update", "apigee.sharedflowrevisions.deploy", "apigee.sharedflowrevisions.undeploy",
    "apigee.keyvaluemaps.create", "apigee.keyvaluemaps.get", "apigee.keyvaluemaps.update", "apigee.keyvaluemaps.delete",
    "apigee.keyvaluemapentries.create", "apigee.keyvaluemapentries.get", "apigee.keyvaluemapentries.update", "apigee.keyvaluemapentries.delete",
    "apigee.targetservers.create", "apigee.targetservers.get", "apigee.targetservers.update", "apigee.targetservers.delete",
]
iam_url = f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}:testIamPermissions"
_, iam_resp = gcp_request("POST", iam_url, {"permissions": REQUIRED_APIGEE_PERMS})
missing_perms = [p for p in REQUIRED_APIGEE_PERMS if p not in set(iam_resp.get("permissions", []))]
if missing_perms:
    raise SystemExit(f"❌ HALT: Missing Apigee IAM permissions: {", ".join(missing_perms)}")
PREFLIGHT["apigee"] = True
print(f"✅ [0.2] Verified all {len(REQUIRED_APIGEE_PERMS)} Apigee IAM permissions.")

# ============================================================================
# 0.3 Verify Service Account Roles, Permissions & <GoogleAccessToken> Bindings + Live Gemini
# ============================================================================
print("\n▶ [0.3] Verify Service Account Roles, Permissions & Bindings + Live Gemini...")
REQUIRED_SA_ROLES = ["roles/aiplatform.user", "roles/iam.serviceAccountTokenCreator"]

def ensure_policy_bindings(get_url: str, set_url: str, required_pairs: list, label: str):
    status, policy = gcp_request("POST", get_url, {})
    if status != 200:
        raise SystemExit(f"❌ HALT: Failed to read {label} IAM policy (HTTP {status}): {policy}")
    bindings = policy.setdefault("bindings", [])
    changed = False
    for member, role in required_pairs:
        entry = next((b for b in bindings if b.get("role") == role and not b.get("condition")), None)
        if entry and member in entry.get("members", []):
            print(f"  ✔ {label} verified: {role} -> {member}")
        else:
            print(f"  ⏳ Binding {label}: {role} -> {member}")
            if entry:
                entry.setdefault("members", []).append(member)
            else:
                bindings.append({"role": role, "members": [member]})
            changed = True
    if changed:
        s_code, s_resp = gcp_request("POST", set_url, {"policy": policy})
        if s_code != 200:
            raise SystemExit(f"❌ HALT: Failed to update {label} IAM policy (HTTP {s_code}): {s_resp}")
        print(f"  ✔ {label} bindings updated.")

# 1. Resolve Project Number & Apigee Service Agent
_, proj_data = gcp_request("GET", f"https://cloudresourcemanager.googleapis.com/v1/projects/{APIGEE_ORG}")
if not proj_data.get("projectNumber"):
    _, proj_data = gcp_request("GET", f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}")
if not proj_data.get("projectNumber"):
    raise SystemExit(f"❌ HALT: Unable to resolve projectNumber for '{PROJECT_ID}': {proj_data}")

PROJECT_NUMBER = str(proj_data["projectNumber"])
APIGEE_SERVICE_AGENT = f"service-{PROJECT_NUMBER}@gcp-sa-apigee.iam.gserviceaccount.com"

# 2. Verify Provided Service Account (or Auto-Provision Default Only When Not Provided)
target_sa = SERVICE_ACCOUNT or f"apigee-ai-gateway-sa@{PROJECT_ID}.iam.gserviceaccount.com"
sa_status, sa_data = gcp_request("GET", f"https://iam.googleapis.com/v1/projects/-/serviceAccounts/{target_sa}")

if SA_PROVIDED:
    if sa_status != 200 or sa_data.get("disabled", False):
        raise SystemExit(f"❌ HALT: Provided SERVICE_ACCOUNT '{SERVICE_ACCOUNT}' is missing or disabled (HTTP {sa_status}).")
    SA_STATUS = "VERIFIED_EXISTING"
    print(f"✅ Verified provided SERVICE_ACCOUNT: {SERVICE_ACCOUNT}")
elif sa_status == 200:
    SERVICE_ACCOUNT, SA_STATUS = target_sa, "REUSED_DEFAULT"
elif sa_status == 404:
    c_status, c_data = gcp_request(
        "POST",
        f"https://iam.googleapis.com/v1/projects/{PROJECT_ID}/serviceAccounts",
        {"accountId": "apigee-ai-gateway-sa", "serviceAccount": {"displayName": "Apigee AI Gateway Service Account"}},
    )
    if c_status not in (200, 201):
        raise SystemExit(f"❌ HALT: Failed to create default service account '{target_sa}' (HTTP {c_status}): {c_data}")
    SERVICE_ACCOUNT, SA_STATUS = c_data.get("email", target_sa), "CREATED_DEFAULT"
    time.sleep(3)
else:
    raise SystemExit(f"❌ HALT: Unable to inspect service account '{target_sa}' (HTTP {sa_status}): {sa_data}")

# 3. Verify & Ensure Project Roles (Gemini + Token Creator) and SA Resource Bindings (<GoogleAccessToken>)
proj_iam_base = f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}"
ensure_policy_bindings(
    f"{proj_iam_base}:getIamPolicy",
    f"{proj_iam_base}:setIamPolicy",
    [(f"serviceAccount:{SERVICE_ACCOUNT}", r) for r in REQUIRED_SA_ROLES],
    "Project role",
)

sa_iam_base = f"https://iam.googleapis.com/v1/projects/-/serviceAccounts/{SERVICE_ACCOUNT}"
ensure_policy_bindings(
    f"{sa_iam_base}:getIamPolicy",
    f"{sa_iam_base}:setIamPolicy",
    [
        (f"serviceAccount:{APIGEE_SERVICE_AGENT}", "roles/iam.serviceAccountTokenCreator"),
        (f"serviceAccount:{SERVICE_ACCOUNT}", "roles/iam.serviceAccountTokenCreator"),
        (f"serviceAccount:{SERVICE_ACCOUNT}", "roles/iam.serviceAccountUser"),
        (f"{caller_member_prefix}:{ACTIVE_IDENTITY}", "roles/iam.serviceAccountUser"),
    ],
    "SA <GoogleAccessToken> binding",
)

os.environ["SERVICE_ACCOUNT"] = SERVICE_ACCOUNT
PREFLIGHT["service_account"] = True
print(f"✅ Service Account ready for Gemini & <GoogleAccessToken>: {SERVICE_ACCOUNT} ({SA_STATUS})")

# 4. Verify Live Gemini Connectivity
payload = {"contents": [{"role": "user", "parts": [{"text": "Reply with READY."}]}], "generationConfig": {"maxOutputTokens": 8, "temperature": 0.0}}
def invoke_gemini(loc: str):
    host = "aiplatform.googleapis.com" if loc == "global" else f"{loc}-aiplatform.googleapis.com"
    url = f"https://{host}/v1/projects/{PROJECT_ID}/locations/{loc}/publishers/google/models/{GEMINI_MODEL}:generateContent"
    s, d = gcp_request("POST", url, payload, headers={"x-goog-user-project": PROJECT_ID})
    return gcp_request("POST", url, payload) if s == 403 else (s, d)

VERTEX_LOCATION, gen_status, gen_data = REGION, 0, {}
for _ in range(12):
    gen_status, gen_data = invoke_gemini(REGION)
    if gen_status == 200:
        VERTEX_LOCATION = REGION
        break
    for fb in [loc for loc in ("global", "us-central1") if loc != REGION]:
        fb_s, fb_d = invoke_gemini(fb)
        if fb_s == 200:
            gen_status, gen_data, VERTEX_LOCATION = fb_s, fb_d, fb
            break
    if gen_status == 200 or gen_status not in (403, 404):
        break
    time.sleep(5)

if gen_status != 200:
    raise SystemExit(f"❌ HALT: Live Gemini call failed (HTTP {gen_status}): {gen_data}")

os.environ["VERTEX_LOCATION"] = VERTEX_LOCATION
PREFLIGHT["gemini"] = True
print(f"✅ [0.3] Live Gemini verified: model={GEMINI_MODEL} | location={VERTEX_LOCATION}")

# ============================================================================
# 0.4 Install AFT CLI & Preflight Summary Gate
# ============================================================================
print("\n▶ [0.4] Install AFT CLI & Preflight Summary Gate...")
os.environ.pop("AFT_REPOSITORY", None)
for extra_bin in [str(pathlib.Path.home() / ".aft" / "bin"), str(pathlib.Path.home() / ".local" / "bin")]:
    if extra_bin not in os.environ["PATH"]:
        os.environ["PATH"] = f"{extra_bin}:{os.environ['PATH']}"

if not shutil.which("aft"):
    subprocess.run("curl -fsSL https://raw.githubusercontent.com/apigee/apigee-templater/main/install.sh | sh", shell=True, check=True)

aft_bin = shutil.which("aft")
if not aft_bin:
    raise SystemExit("❌ HALT: 'aft' binary not found on PATH after installation.")
aft_ver = subprocess.run([aft_bin, "--version"], capture_output=True, text=True, check=True).stdout.strip()
PREFLIGHT["aft"] = True
print(f"✅ AFT CLI ready: {aft_ver} ({aft_bin})")

# 2. Clone GitHub Repository (local templates & features for AFT)
REPO_URL = "https://github.com/luiscuellarh/apigee-unified-ai-gateway.git"
REPO_BRANCH = "main"
REPO_DIR = pathlib.Path.cwd() / "apigee-unified-ai-gateway"

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True, capture_output=True, text=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=True, capture_output=True, text=True)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True, capture_output=True, text=True)

if not (REPO_DIR / "templates").is_dir() or not (REPO_DIR / "features").is_dir():
    raise SystemExit(f"❌ HALT: Cloned repository at '{REPO_DIR}' is missing 'templates/' or 'features/' directories.")

WORKSPACE_DIR = REPO_DIR
os.environ.update({"REPO_URL": REPO_URL, "REPO_BRANCH": REPO_BRANCH, "REPO_DIR": str(REPO_DIR), "WORKSPACE_DIR": str(WORKSPACE_DIR)})
PREFLIGHT["repo"] = True
num_templates = len(list((REPO_DIR / "templates").glob("*.yaml")))
num_features = len(list((REPO_DIR / "features").glob("*.yaml")))
print(f"✅ Cloned GitHub repo ({REPO_BRANCH}): {REPO_DIR} ({num_templates} templates, {num_features} features)")

# 3. Preflight Summary Gate
failed_gates = [k for k, v in PREFLIGHT.items() if not v]
if failed_gates:
    raise SystemExit(f"❌ HALT: Preflight incomplete. Failed gate(s): {", ".join(failed_gates)}")

print("=" * 72)
for label, val in [
    ("GCP Project / No.",  f"{PROJECT_ID} ({PROJECT_NUMBER})"),
    ("Apigee Org / Env",   f"{APIGEE_ORG} / {APIGEE_ENV} ({ENV_GROUP})"),
    ("Gateway Hostname",   HOSTNAME),
    ("Active Identity",    ACTIVE_IDENTITY),
    ("Service Account",    f"{SERVICE_ACCOUNT} ({SA_STATUS}, <GoogleAccessToken> ready)"),
    ("Gemini Readiness",   f"PASS ({GEMINI_MODEL} @ {VERTEX_LOCATION})"),
    ("AFT Toolkit",        f"READY ({aft_ver})"),
    ("Local Repo (AFT)",   f"{REPO_DIR} (branch={REPO_BRANCH}, {num_templates} templates, {num_features} features)"),
]:
    print(f"{label:<20}: {val}")
print("=" * 72)
print("PREFLIGHT PASSED — safe to proceed")


## Step 1 — Deploy MCP Proxy for Loans, API Product & Developer App
- **Purpose:** Deploy the Model Context Protocol (MCP) Gateway proxy for the Loans service (`mcp-gateway-loan-product`), register the **AI Loan Agent Package** (`ai-loan-agent-package`) enforcing MCP operation-level allow-listing and quota (`tools/list` and `tools/call/calculate_monthly_installment` at 30 req/min), and onboard the **Loan Agent App** (`loan-agent@example.com`).
- **AFT Deployments:**
  1. **MCP Loan Proxy:** `aft -i templates/mcp-gateway-loan-product.yaml --organization $APIGEE_ORG --environment $APIGEE_ENVIRONMENT --sa $SERVICE_ACCOUNT`
  2. **API Product / Package:** `aft -i config/products/ai-loan-agent-package.yaml --organization $APIGEE_ORG --environment $APIGEE_ENVIRONMENT --sa $SERVICE_ACCOUNT`
  3. **Developer & App:** `aft -i config/apps/user_apps_mcp_loan.yaml --organization $APIGEE_ORG --environment $APIGEE_ENVIRONMENT --sa $SERVICE_ACCOUNT`
- **Dependencies:** Step 0 (Preflight)

In [ ]:
#@title 1.1 Deploy MCP Proxy for Loans, API Product & Developer App via AFT
# 1) Deploys templates/mcp-gateway-loan-product.yaml via AFT and waits until READY
# 2) Deploys config/products/ai-loan-agent-package.yaml via AFT
# 3) Deploys config/apps/user_apps_mcp_loan.yaml via AFT
# (Also ensures features/auth-oauth-server.yaml is active in APIGEE_ENVIRONMENT for OAuth 2.0 token generation)

os.environ.pop("AFT_REPOSITORY", None)
APIGEE_ENVIRONMENT = APIGEE_ENV
os.environ["APIGEE_ENVIRONMENT"] = APIGEE_ENVIRONMENT

TEMPLATE_LOAN_REL_PATH = "templates/mcp-gateway-loan-product.yaml"
FEATURE_LOAN_ENDPOINT_REL_PATH = "features/mcp-loan-endpoint.yaml"
FEATURE_OAUTH_QUOTA_REL_PATH = "features/api-quota-policy-spike-oauth.yaml"
FEATURE_OAUTH_SERVER_REL_PATH = "features/auth-oauth-server.yaml"
PRODUCT_LOAN_REL_PATH = "config/products/ai-loan-agent-package.yaml"
APPS_LOAN_REL_PATH = "config/apps/user_apps_mcp_loan.yaml"

required_rel_paths = [
    TEMPLATE_LOAN_REL_PATH,
    FEATURE_LOAN_ENDPOINT_REL_PATH,
    FEATURE_OAUTH_QUOTA_REL_PATH,
    FEATURE_OAUTH_SERVER_REL_PATH,
    PRODUCT_LOAN_REL_PATH,
    APPS_LOAN_REL_PATH,
]
if any(not (REPO_DIR / p).is_file() for p in required_rel_paths):
    print(f"🔄 Fetching latest files from origin/{REPO_BRANCH}...")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], capture_output=True, text=True)
    for rel_p in required_rel_paths:
        if not (REPO_DIR / rel_p).is_file():
            subprocess.run(["git", "-C", str(REPO_DIR), "checkout", f"origin/{REPO_BRANCH}", "--", rel_p], capture_output=True, text=True)

for rel_p in required_rel_paths:
    if not (REPO_DIR / rel_p).is_file():
        raise SystemExit(f"❌ HALT: Required file not found at '{REPO_DIR / rel_p}'. Ensure it is committed to the repo or re-run Step 0.")

def run_aft_deploy(rel_path: str, label: str) -> None:
    cmd = [
        aft_bin,
        "-i", rel_path,
        "--organization", APIGEE_ORG,
        "--environment", APIGEE_ENVIRONMENT,
        "--sa", SERVICE_ACCOUNT,
    ]
    print(f"\n▶ {label}: {' '.join(cmd)}")
    res = subprocess.run(cmd, cwd=str(REPO_DIR), capture_output=True, text=True)
    if res.returncode != 0:
        err_out = (res.stderr or res.stdout).strip()
        raise SystemExit(f"❌ HALT: AFT deployment failed for '{rel_path}' (exit {res.returncode}):\n{err_out}")
    if res.stdout.strip():
        print(res.stdout.strip())

# Ensure OAuth 2.0 token server (auth-oauth-server) is deployed in APIGEE_ENVIRONMENT so /v1/oauth/token can mint tokens
OAUTH_PROXY_NAME = "auth-oauth-server"
oauth_s, oauth_d = gcp_request("GET", f"{APIGEE_BASE}/environments/{APIGEE_ENVIRONMENT}/apis/{OAUTH_PROXY_NAME}/deployments")
if oauth_s != 200 or not oauth_d.get("deployments"):
    run_aft_deploy(FEATURE_OAUTH_SERVER_REL_PATH, f"[0/3] Deploying OAuth 2.0 Token Server ({OAUTH_PROXY_NAME})")
wait_for_proxy_ready(OAUTH_PROXY_NAME, APIGEE_ENVIRONMENT)

# 1. Deploy the MCP Loan Proxy
PROXY_NAME_MCP_LOAN = "mcp-gateway-loan-product"
run_aft_deploy(TEMPLATE_LOAN_REL_PATH, "[1/3] Deploying MCP Loan Proxy")
deployed_loan_rev = wait_for_proxy_ready(PROXY_NAME_MCP_LOAN, APIGEE_ENVIRONMENT)

# 2. Deploy the API Product / Package (ai-loan-agent-package)
prod_loan_path = REPO_DIR / PRODUCT_LOAN_REL_PATH
prod_loan_text = prod_loan_path.read_text(encoding="utf-8")
if not re.search(rf"(?m)^\s*-\s*{re.escape(APIGEE_ENVIRONMENT)}\s*$", prod_loan_text):
    prod_loan_text = re.sub(
        r"(?m)^(\s*environments:\s*\n)",
        rf"\g<1>  - {APIGEE_ENVIRONMENT}\n",
        prod_loan_text,
        count=1,
    )
    prod_loan_path.write_text(prod_loan_text, encoding="utf-8")

run_aft_deploy(PRODUCT_LOAN_REL_PATH, "[2/3] Deploying API Product / Package")
print(f"✅ Deployed API Product: {PRODUCT_LOAN_REL_PATH}")

# 3. Deploy the Developer and App (user_apps_mcp_loan.yaml)
run_aft_deploy(APPS_LOAN_REL_PATH, "[3/3] Deploying Developer & App")
print(f"✅ Deployed Developer & App: {APPS_LOAN_REL_PATH}")

OAUTH_TOKEN_ENDPOINT = f"https://{HOSTNAME}/v1/oauth/token"
MCP_LOAN_ENDPOINT = f"https://{HOSTNAME}/loanservice/mcp"
os.environ.update({
    "PROXY_NAME_MCP_LOAN": PROXY_NAME_MCP_LOAN,
    "OAUTH_TOKEN_ENDPOINT": OAUTH_TOKEN_ENDPOINT,
    "MCP_LOAN_ENDPOINT": MCP_LOAN_ENDPOINT,
})

print("=" * 72)
print(f"✅ Deployed MCP Proxy : {PROXY_NAME_MCP_LOAN} (rev={deployed_loan_rev})")
print(f"✅ API Product        : ai-loan-agent-package")
print(f"✅ Developer App      : Loan Agent App (loan-agent@example.com)")
print(f"✅ OAuth Token URL    : {OAUTH_TOKEN_ENDPOINT}")
print(f"✅ MCP Loan Endpoint  : {MCP_LOAN_ENDPOINT}")
print("=" * 72)

In [ ]:
#@title 1.2 Test MCP Loan Proxy (Standard MCP Protocol & Dynamic Tool UX)
#@markdown ### OAuth 2.0 App Credentials (`loan-agent-app`)
CLIENT_ID = "loan-agent-app-key-123" #@param {type:"string"}
CLIENT_SECRET = "loan-agent-app-secret-123" #@param {type:"string"}

from dataclasses import dataclass
import html, json, time
from typing import Any, Dict, List, Optional, Tuple
from IPython.display import display, HTML
import ipywidgets as widgets
import requests, urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

_host = str(globals().get("HOSTNAME", "")).strip()
if not _host:
  raise RuntimeError("❌ HOSTNAME is empty. Please run Step 0 first.")

_org = str(globals().get("APIGEE_ORG", "")).strip()
_env = str(globals().get("APIGEE_ENV", "")).strip()
PROXY_NAME = "mcp-gateway-loan-product"
ACTIVE_REVISION = wait_for_proxy_ready(PROXY_NAME, _env) if "wait_for_proxy_ready" in globals() else "latest"
OAUTH_TOKEN_URL = f"https://{_host}/v1/oauth/token"
MCP_ENDPOINT_URL = f"https://{_host}/loanservice/mcp"

@dataclass
class StepMetric:
  name: str
  status_code: int
  latency_ms: int
  detail: str
  ok: bool = True

@dataclass
class MCPCallResult:
  tool_name: str
  request_headers: Dict[str, str]
  request_payload: Dict[str, Any]
  status_code: int
  response_headers: Dict[str, str]
  response_body: Dict[str, Any]
  latency_ms: int
  session_id: Optional[str]
  reinitialized: bool = False

class StandardMCPClient:
  """Spec-compliant Model Context Protocol (MCP) Streamable HTTP client."""
  PROTOCOL_VERSION = "2024-11-05"

  def __init__(self, oauth_url: str, mcp_url: str, client_id: str, client_secret: str):
    self.oauth_url = oauth_url
    self.mcp_url = mcp_url
    self.client_id = client_id.strip()
    self.client_secret = client_secret.strip()
    self.access_token: Optional[str] = None
    self.token_expires_in: Optional[Any] = None
    self.session_id: Optional[str] = None
    self.protocol_version: str = self.PROTOCOL_VERSION
    self.server_info: Dict[str, Any] = {}
    self.server_capabilities: Dict[str, Any] = {}
    self.tools: List[Dict[str, Any]] = []
    self.handshake_metrics: List[StepMetric] = []
    self._rpc_id = 0
    self._http = requests.Session()
    self._http.verify = False

  def _next_id(self) -> int:
    self._rpc_id += 1
    return self._rpc_id

  @property
  def masked_token(self) -> str:
    if not self.access_token:
      return "none"
    return "***" if len(self.access_token) <= 14 else f"{self.access_token[:8]}...{self.access_token[-6:]}"

  def _mcp_headers(self, include_session: bool = True, mask_auth: bool = False) -> Dict[str, str]:
    token_str = self.masked_token if mask_auth else (self.access_token or "")
    hdrs = {
        "Authorization": f"Bearer {token_str}",
        "Content-Type": "application/json",
        "Accept": "application/json, text/event-stream",
    }
    if include_session and self.session_id:
      hdrs["mcp-session-id"] = self.session_id
      hdrs["mcp-protocol-version"] = self.protocol_version
    return hdrs

  @staticmethod
  def _parse_response(resp: requests.Response) -> Dict[str, Any]:
    ctype = (resp.headers.get("Content-Type") or "").lower()
    raw = (resp.text or "").strip()
    if not raw:
      return {}
    if "text/event-stream" in ctype or raw.startswith(("event:", "data:")):
      last_event: Dict[str, Any] = {}
      for line in raw.splitlines():
        s = line.strip()
        if s.startswith("data:") and s[5:].strip():
          try:
            last_event = json.loads(s[5:].strip())
          except ValueError:
            pass
      if last_event:
        return last_event
    try:
      return resp.json()
    except ValueError:
      return {"raw_text": raw}

  def _capture_session_id(self, resp: requests.Response) -> None:
    for k, v in resp.headers.items():
      if k.lower() == "mcp-session-id" and v:
        self.session_id = v.strip()
        return

  def mint_oauth_token(self) -> StepMetric:
    t0 = time.perf_counter()
    resp = self._http.post(
        self.oauth_url,
        auth=(self.client_id, self.client_secret),
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        data={"grant_type": "client_credentials"},
        timeout=20,
    )
    ms = int((time.perf_counter() - t0) * 1000)
    if resp.status_code != 200:
      raise RuntimeError(f"OAuth 2.0 token request failed (HTTP {resp.status_code}, {ms}ms): {resp.text[:400]}")
    data = resp.json()
    self.access_token = data.get("access_token", "")
    self.token_expires_in = data.get("expires_in", "N/A")
    return StepMetric("1. OAuth 2.0 Token", resp.status_code, ms, f"token={self.masked_token} (TTL {self.token_expires_in}s)")

  def initialize_session(self) -> Tuple[StepMetric, StepMetric]:
    if not self.access_token:
      self.mint_oauth_token()
    self.session_id = None
    init_payload = {
        "jsonrpc": "2.0",
        "id": self._next_id(),
        "method": "initialize",
        "params": {
            "protocolVersion": self.PROTOCOL_VERSION,
            "capabilities": {},
            "clientInfo": {"name": "apigee-mcp-gateway-colab", "version": "1.0.0"},
        },
    }
    t0 = time.perf_counter()
    resp = self._http.post(self.mcp_url, headers=self._mcp_headers(include_session=False), json=init_payload, timeout=30)
    init_ms = int((time.perf_counter() - t0) * 1000)
    self._capture_session_id(resp)
    body = self._parse_response(resp)
    if resp.status_code != 200 or "error" in body:
      raise RuntimeError(f"MCP initialize failed (HTTP {resp.status_code}, {init_ms}ms): {json.dumps(body, ensure_ascii=False)}")
    result = body.get("result", {})
    self.protocol_version = result.get("protocolVersion", self.PROTOCOL_VERSION)
    self.server_info = result.get("serverInfo", {})
    self.server_capabilities = result.get("capabilities", {})
    srv_name = self.server_info.get("name", "mcp-server")
    srv_ver = self.server_info.get("version", "")
    init_metric = StepMetric("2. MCP initialize", resp.status_code, init_ms, f"{srv_name} v{srv_ver} • proto {self.protocol_version}")

    notif_payload = {"jsonrpc": "2.0", "method": "notifications/initialized"}
    t1 = time.perf_counter()
    notif_resp = self._http.post(self.mcp_url, headers=self._mcp_headers(include_session=True), json=notif_payload, timeout=20)
    notif_ms = int((time.perf_counter() - t1) * 1000)
    self._capture_session_id(notif_resp)
    notif_metric = StepMetric("3. notifications/initialized", notif_resp.status_code, notif_ms, f"session={self.session_id or 'stateless'}")
    return init_metric, notif_metric

  def list_tools(self) -> StepMetric:
    if not self.session_id and not self.server_info:
      self.initialize_session()
    payload = {"jsonrpc": "2.0", "id": self._next_id(), "method": "tools/list"}
    t0 = time.perf_counter()
    resp = self._http.post(self.mcp_url, headers=self._mcp_headers(include_session=True), json=payload, timeout=30)
    ms = int((time.perf_counter() - t0) * 1000)
    self._capture_session_id(resp)
    body = self._parse_response(resp)
    if resp.status_code != 200 or "error" in body:
      raise RuntimeError(f"MCP tools/list failed (HTTP {resp.status_code}, {ms}ms): {json.dumps(body, ensure_ascii=False)}")
    self.tools = body.get("result", {}).get("tools", [])
    return StepMetric("4. MCP tools/list", resp.status_code, ms, f"{len(self.tools)} tool(s) discovered")

  def bootstrap(self, remint_token: bool = True) -> List[Dict[str, Any]]:
    self.handshake_metrics.clear()
    if remint_token or not self.access_token:
      self.handshake_metrics.append(self.mint_oauth_token())
    m_init, m_notif = self.initialize_session()
    m_list = self.list_tools()
    self.handshake_metrics.extend([m_init, m_notif, m_list])
    return self.tools

  def call_tool(self, name: str, arguments: Dict[str, Any]) -> MCPCallResult:
    if not isinstance(arguments, dict):
      raise ValueError(f"MCP tools/call 'arguments' must be a dict, got {type(arguments).__name__}")
    payload = {"jsonrpc": "2.0", "id": self._next_id(), "method": "tools/call", "params": {"name": name, "arguments": arguments}}
    t0 = time.perf_counter()
    resp = self._http.post(self.mcp_url, headers=self._mcp_headers(include_session=True), json=payload, timeout=30)
    ms = int((time.perf_counter() - t0) * 1000)
    body = self._parse_response(resp)
    reinitialized = False
    err_msg = str(body.get("error", {}).get("message", "")).lower() if isinstance(body, dict) else ""
    if resp.status_code == 404 or "before initialization" in err_msg or "session" in err_msg:
      self.initialize_session()
      reinitialized = True
      t0 = time.perf_counter()
      resp = self._http.post(self.mcp_url, headers=self._mcp_headers(include_session=True), json=payload, timeout=30)
      ms = int((time.perf_counter() - t0) * 1000)
      body = self._parse_response(resp)
    interesting_resp_hdrs = {k: v for k, v in resp.headers.items() if k.lower() in ("content-type", "mcp-session-id", "date", "server", "via")}
    return MCPCallResult(
        tool_name=name,
        request_headers=self._mcp_headers(include_session=True, mask_auth=True),
        request_payload=payload,
        status_code=resp.status_code,
        response_headers=interesting_resp_hdrs,
        response_body=body,
        latency_ms=ms,
        session_id=self.session_id,
        reinitialized=reinitialized,
    )

MCP_UX_CSS = """
<style>
.mcp-ux-root { font-family: 'Google Sans', 'Roboto', -apple-system, sans-serif; color: #1f2937; max-width: 980px; margin: 6px 0 12px 0; }
.mcp-hero { background: linear-gradient(135deg, #0f172a 0%, #1e3a8a 100%); color: #f8fafc; border-radius: 12px; padding: 16px 20px; box-shadow: 0 4px 14px rgba(15,23,42,0.14); margin-bottom: 12px; }
.mcp-hero-top { display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap; gap: 8px; margin-bottom: 10px; }
.mcp-hero-title { font-size: 16px; font-weight: 700; letter-spacing: 0.2px; display: flex; align-items: center; gap: 8px; }
.mcp-pill { display: inline-flex; align-items: center; gap: 5px; font-size: 11px; font-weight: 600; padding: 3px 10px; border-radius: 999px; background: rgba(255,255,255,0.14); color: #e2e8f0; border: 1px solid rgba(255,255,255,0.2); }
.mcp-pill-green { background: rgba(16,185,129,0.2); color: #6ee7b7; border-color: rgba(16,185,129,0.4); }
.mcp-meta-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(210px, 1fr)); gap: 8px; font-size: 12px; background: rgba(15,23,42,0.45); padding: 10px 12px; border-radius: 8px; border: 1px solid rgba(255,255,255,0.1); }
.mcp-meta-item span { color: #94a3b8; display: block; font-size: 10px; text-transform: uppercase; letter-spacing: 0.5px; }
.mcp-meta-item code { color: #f1f5f9; font-family: 'JetBrains Mono', monospace; font-size: 11.5px; word-break: break-all; }
.mcp-steps-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(205px, 1fr)); gap: 8px; margin-top: 10px; }
.mcp-step-card { background: rgba(255,255,255,0.08); border: 1px solid rgba(255,255,255,0.15); border-radius: 8px; padding: 8px 10px; }
.mcp-step-hdr { display: flex; justify-content: space-between; align-items: center; font-size: 11.5px; font-weight: 600; color: #f8fafc; margin-bottom: 3px; }
.mcp-step-ms { font-size: 10.5px; color: #38bdf8; font-family: monospace; }
.mcp-step-sub { font-size: 11px; color: #cbd5e1; font-family: monospace; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }
.mcp-tool-card { background: #ffffff; border: 1px solid #e2e8f0; border-left: 4px solid #2563eb; border-radius: 10px; padding: 12px 16px; margin: 6px 0; box-shadow: 0 1px 3px rgba(0,0,0,0.04); }
.mcp-tool-hdr { display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap; gap: 8px; margin-bottom: 6px; }
.mcp-tool-name { font-family: 'JetBrains Mono', monospace; font-size: 14px; font-weight: 700; color: #1e40af; background: #eff6ff; padding: 3px 10px; border-radius: 6px; border: 1px solid #bfdbfe; }
.mcp-tool-desc { font-size: 13px; color: #334155; line-height: 1.45; margin: 4px 0 8px 0; }
.mcp-schema-details summary { font-size: 11.5px; color: #475569; cursor: pointer; user-select: none; font-weight: 600; }
.mcp-schema-pre { background: #0f172a; color: #e2e8f0; padding: 10px 12px; border-radius: 8px; font-size: 11.5px; font-family: 'JetBrains Mono', monospace; overflow-x: auto; margin-top: 6px; }
.mcp-param-meta { line-height: 1.35; }
.mcp-badge-req { background: #fef2f2; color: #dc2626; border: 1px solid #fecaca; font-size: 10px; font-weight: 700; padding: 1px 6px; border-radius: 4px; text-transform: uppercase; margin-right: 5px; }
.mcp-badge-opt { background: #f1f5f9; color: #64748b; border: 1px solid #cbd5e1; font-size: 10px; font-weight: 600; padding: 1px 6px; border-radius: 4px; text-transform: uppercase; margin-right: 5px; }
.mcp-badge-type { background: #eef2ff; color: #4f46e5; border: 1px solid #c7d2fe; font-size: 10px; font-family: monospace; padding: 1px 6px; border-radius: 4px; margin-right: 6px; }
.mcp-res-card { border-radius: 10px; border: 1px solid #e2e8f0; background: #ffffff; overflow: hidden; box-shadow: 0 2px 8px rgba(15,23,42,0.06); margin-top: 4px; }
.mcp-res-banner { padding: 10px 16px; display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap; gap: 8px; font-weight: 700; font-size: 13px; }
.mcp-res-ok { background: #ecfdf5; color: #065f46; border-bottom: 1px solid #a7f3d0; }
.mcp-res-err { background: #fef2f2; color: #991b1b; border-bottom: 1px solid #fecaca; }
.mcp-res-warn { background: #fffbeb; color: #92400e; border-bottom: 1px solid #fde68a; }
.mcp-res-body { padding: 14px 16px; }
.mcp-output-box { background: #f8fafc; border: 1px solid #cbd5e1; border-left: 4px solid #10b981; border-radius: 8px; padding: 12px 14px; margin-bottom: 12px; }
.mcp-output-title { font-size: 11.5px; font-weight: 700; text-transform: uppercase; letter-spacing: 0.4px; color: #047857; margin-bottom: 6px; }
.mcp-kv-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(170px, 1fr)); gap: 8px; margin-bottom: 8px; }
.mcp-kv-card { background: #ffffff; border: 1px solid #e2e8f0; border-radius: 6px; padding: 8px 10px; }
.mcp-kv-key { font-size: 10.5px; color: #64748b; text-transform: uppercase; letter-spacing: 0.3px; }
.mcp-kv-val { font-size: 14px; font-weight: 700; color: #0f172a; font-family: 'JetBrains Mono', monospace; margin-top: 2px; }
.mcp-wire-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(360px, 1fr)); gap: 12px; }
.mcp-wire-panel { background: #0f172a; color: #e2e8f0; border-radius: 8px; padding: 10px 12px; font-family: 'JetBrains Mono', monospace; font-size: 11.5px; overflow-x: auto; }
.mcp-wire-hdr { color: #94a3b8; font-size: 11px; font-weight: 700; text-transform: uppercase; letter-spacing: 0.5px; border-bottom: 1px solid #1e293b; padding-bottom: 6px; margin-bottom: 8px; display: flex; justify-content: space-between; }
.mcp-wire-pre { margin: 0; white-space: pre-wrap; word-break: break-word; color: #f1f5f9; }
</style>
"""

def _render_dashboard_html(client: StandardMCPClient) -> str:
  steps_html = "".join(
      f"<div class='mcp-step-card'><div class='mcp-step-hdr'><span>✅ {html.escape(m.name)}</span>"
      f"<span class='mcp-step-ms'>HTTP {m.status_code} • {m.latency_ms}ms</span></div>"
      f"<div class='mcp-step-sub' title='{html.escape(m.detail)}'>{html.escape(m.detail)}</div></div>"
      for m in client.handshake_metrics
  )
  srv_title = client.server_info.get("name", "MCP Server")
  srv_ver = client.server_info.get("version", "")
  return (
      f"{MCP_UX_CSS}<div class='mcp-ux-root'><div class='mcp-hero'>"
      f"<div class='mcp-hero-top'><div class='mcp-hero-title'>⚡ Apigee MCP Gateway — Live Session & Tool Explorer</div>"
      f"<div><span class='mcp-pill mcp-pill-green'>● SESSION READY</span> "
      f"<span class='mcp-pill'>Server: {html.escape(srv_title)} {html.escape(srv_ver)}</span> "
      f"<span class='mcp-pill'>MCP {html.escape(client.protocol_version)}</span> "
      f"<span class='mcp-pill'>Proxy: {html.escape(PROXY_NAME)} (rev {html.escape(str(ACTIVE_REVISION))})</span></div></div>"
      f"<div class='mcp-meta-grid'>"
      f"<div class='mcp-meta-item'><span>OAuth 2.0 Token Endpoint</span><code>{html.escape(client.oauth_url)}</code></div>"
      f"<div class='mcp-meta-item'><span>MCP Gateway Endpoint</span><code>{html.escape(client.mcp_url)}</code></div>"
      f"<div class='mcp-meta-item'><span>Client ID &amp; Token</span><code>{html.escape(client.client_id)} ({html.escape(client.masked_token)})</code></div>"
      f"<div class='mcp-meta-item'><span>Active MCP Session ID</span><code>{html.escape(client.session_id or 'stateless')}</code></div></div>"
      f"<div class='mcp-steps-grid'>{steps_html}</div></div></div>"
  )

def _render_tool_card_html(tool_def: Dict[str, Any]) -> str:
  t_name = tool_def.get("name", "unnamed")
  t_desc = (tool_def.get("description") or "No description provided by MCP server.").strip()
  schema = tool_def.get("inputSchema") or {}
  props = schema.get("properties") or {}
  req_set = set(schema.get("required") or [])
  schema_json = json.dumps(schema, indent=2, ensure_ascii=False)
  return (
      f"<div class='mcp-ux-root'><div class='mcp-tool-card'><div class='mcp-tool-hdr'>"
      f"<div><span class='mcp-tool-name'>🔧 {html.escape(t_name)}</span></div>"
      f"<div style='font-size:12px;color:#475569;font-weight:600;'>{len(props)} parameter(s) • {len(req_set)} required</div></div>"
      f"<div class='mcp-tool-desc'>{html.escape(t_desc)}</div>"
      f"<details class='mcp-schema-details'><summary>📐 Inspect Discovered <code>inputSchema</code> from <code>tools/list</code></summary>"
      f"<pre class='mcp-schema-pre'>{html.escape(schema_json)}</pre></details></div></div>"
  )

def _render_call_result_html(res: MCPCallResult) -> str:
  body = res.response_body
  has_rpc_error = isinstance(body, dict) and "error" in body
  result_obj = body.get("result", {}) if isinstance(body, dict) else {}
  is_tool_error = isinstance(result_obj, dict) and bool(result_obj.get("isError"))
  if res.status_code == 200 and not has_rpc_error and not is_tool_error:
    banner_cls, status_label = "mcp-res-ok", f"✅ ALLOWED BY APIGEE MCP GATEWAY • HTTP {res.status_code} OK"
  elif res.status_code in (401, 403, 429):
    banner_cls, status_label = "mcp-res-err", f"🛡️ BLOCKED BY APIGEE MCP GATEWAY • HTTP {res.status_code}"
  else:
    banner_cls, status_label = "mcp-res-warn", f"⚠️ MCP SERVER / JSON-RPC ERROR • HTTP {res.status_code}"
  reinit_badge = " • 🔄 Session Auto-Reinitialized" if res.reinitialized else ""
  meta_right = f"tool={html.escape(res.tool_name)} • latency={res.latency_ms}ms • session={html.escape(res.session_id or 'stateless')}{reinit_badge}"

  content_blocks_html = ""
  if res.status_code == 200 and not has_rpc_error:
    content_items = result_obj.get("content", []) if isinstance(result_obj, dict) else []
    rendered_items = []
    for item in content_items:
      if isinstance(item, dict) and item.get("type") == "text":
        txt = item.get("text", "")
        try:
          parsed_json = json.loads(txt)
          if isinstance(parsed_json, dict):
            kv_cards = "".join(
                f"<div class='mcp-kv-card'><div class='mcp-kv-key'>{html.escape(str(k))}</div>"
                f"<div class='mcp-kv-val'>{html.escape(str(v))}</div></div>"
                for k, v in parsed_json.items() if not isinstance(v, (dict, list))
            )
            kv_sec = f"<div class='mcp-kv-grid'>{kv_cards}</div>" if kv_cards else ""
            rendered_items.append(f"{kv_sec}<pre style='margin:0;font-family:monospace;font-size:12px;color:#1e293b;'>{html.escape(json.dumps(parsed_json, indent=2, ensure_ascii=False))}</pre>")
          else:
            rendered_items.append(f"<pre style='margin:0;font-family:monospace;font-size:12.5px;color:#0f172a;'>{html.escape(json.dumps(parsed_json, indent=2, ensure_ascii=False))}</pre>")
        except ValueError:
          rendered_items.append(f"<pre style='margin:0;font-family:monospace;font-size:12.5px;color:#0f172a;white-space:pre-wrap;'>{html.escape(txt)}</pre>")
      else:
        rendered_items.append(f"<pre style='margin:0;font-family:monospace;font-size:12px;'>{html.escape(json.dumps(item, indent=2, ensure_ascii=False))}</pre>")
    if rendered_items:
      box_border = "#f59e0b" if is_tool_error else "#10b981"
      title_str = "⚠️ Tool Execution Error Output (result.content)" if is_tool_error else "🎯 MCP Tool Output (result.content)"
      content_blocks_html = f"<div class='mcp-output-box' style='border-left-color:{box_border};'><div class='mcp-output-title'>{title_str}</div>{''.join(rendered_items)}</div>"
  elif has_rpc_error:
    err_pretty = json.dumps(body["error"], indent=2, ensure_ascii=False)
    content_blocks_html = f"<div class='mcp-output-box' style='border-left-color:#ef4444;background:#fef2f2;'><div class='mcp-output-title' style='color:#991b1b;'>⚠️ JSON-RPC 2.0 Error Object</div><pre style='margin:0;font-family:monospace;font-size:12px;color:#7f1d1d;'>{html.escape(err_pretty)}</pre></div>"

  req_wire = json.dumps({"headers": res.request_headers, "body": res.request_payload}, indent=2, ensure_ascii=False)
  resp_wire = json.dumps({"status": res.status_code, "headers": res.response_headers, "body": res.response_body}, indent=2, ensure_ascii=False)
  return (
      f"<div class='mcp-ux-root'><div class='mcp-res-card'><div class='mcp-res-banner {banner_cls}'>"
      f"<span>{status_label}</span><span style='font-family:monospace;font-size:11.5px;'>{meta_right}</span></div>"
      f"<div class='mcp-res-body'>{content_blocks_html}<div class='mcp-wire-grid'>"
      f"<div class='mcp-wire-panel'><div class='mcp-wire-hdr'><span>📤 MCP JSON-RPC 2.0 Request</span><span>POST /loanservice/mcp</span></div><pre class='mcp-wire-pre'>{html.escape(req_wire)}</pre></div>"
      f"<div class='mcp-wire-panel'><div class='mcp-wire-hdr'><span>📥 MCP JSON-RPC 2.0 Response</span><span>HTTP {res.status_code} ({res.latency_ms}ms)</span></div><pre class='mcp-wire-pre'>{html.escape(resp_wire)}</pre></div>"
      f"</div></div></div></div>"
  )

mcp_client = StandardMCPClient(OAUTH_TOKEN_URL, MCP_ENDPOINT_URL, CLIENT_ID, CLIENT_SECRET)
discovered_tools = mcp_client.bootstrap(remint_token=True)
TOOLS_BY_NAME: Dict[str, Dict[str, Any]] = {t["name"]: t for t in discovered_tools if t.get("name")}

dashboard_html = widgets.HTML(value=_render_dashboard_html(mcp_client))
tool_card_html = widgets.HTML()
tool_options = [(f"🔧 {t['name']}", t["name"]) for t in discovered_tools if t.get("name")]
tool_dropdown = widgets.Dropdown(options=tool_options, value=tool_options[0][1] if tool_options else None, description="Select Tool:", style={"description_width": "95px"}, layout=widgets.Layout(width="440px", height="36px"))
refresh_btn = widgets.Button(description="🔄 Re-sync tools/list", button_style="info", tooltip="Re-run MCP initialize + tools/list handshake", layout=widgets.Layout(width="180px", height="36px"))
remint_btn = widgets.Button(description="🔑 New OAuth Token", button_style="", tooltip="Mint a fresh OAuth 2.0 access token and re-initialize MCP session", layout=widgets.Layout(width="175px", height="36px"))
params_box = widgets.VBox(layout=widgets.Layout(background="#f8fafc", border="1px solid #e2e8f0", padding="12px 16px", margin="6px 0 10px 0", width="100%", max_width="980px"))
call_btn = widgets.Button(description="▶ Execute tools/call", button_style="primary", tooltip="Send JSON-RPC 2.0 tools/call request over active MCP session", layout=widgets.Layout(width="220px", height="40px"))
clear_btn = widgets.Button(description="🧹 Clear Output", button_style="", layout=widgets.Layout(width="140px", height="40px"))
output_area = widgets.Output(layout=widgets.Layout(width="100%", max_width="980px"))
_param_widgets: Dict[str, Tuple[widgets.Widget, Dict[str, Any], bool]] = {}

def _resolve_json_type(prop_spec: Dict[str, Any]) -> str:
  raw_type = prop_spec.get("type", "string")
  return next((t for t in raw_type if t != "null"), "string") if isinstance(raw_type, list) else str(raw_type)

def _initial_value_from_schema(prop_spec: Dict[str, Any]) -> Any:
  if "default" in prop_spec and prop_spec["default"] is not None:
    return prop_spec["default"]
  if "enum" in prop_spec and prop_spec["enum"]:
    return prop_spec["enum"][0]
  p_type = _resolve_json_type(prop_spec)
  return False if p_type == "boolean" else (0 if p_type == "integer" else (0.0 if p_type == "number" else ("[]" if p_type == "array" else ("{}" if p_type == "object" else ""))))

def _rebuild_parameter_form(tool_name: Optional[str]) -> None:
  _param_widgets.clear()
  if not tool_name or tool_name not in TOOLS_BY_NAME:
    tool_card_html.value = "<div class='mcp-ux-root'><i>No MCP tools available.</i></div>"
    params_box.children = []
    return
  tool_def = TOOLS_BY_NAME[tool_name]
  tool_card_html.value = _render_tool_card_html(tool_def)
  schema = tool_def.get("inputSchema") or {}
  props = schema.get("properties") or {}
  req_set = set(schema.get("required") or [])
  if not props:
    params_box.children = [widgets.HTML("<div style='font-size:12.5px;color:#475569;'>✨ This tool accepts no arguments (<code>\"arguments\": {}</code>).</div>")]
    return
  form_rows: List[widgets.Widget] = [widgets.HTML("<div style='font-size:12px;font-weight:700;text-transform:uppercase;letter-spacing:0.5px;color:#475569;margin-bottom:6px;'>📝 Dynamic Tool Parameters (Generated from <code>inputSchema</code>)</div>")]
  for p_name, p_spec in props.items():
    is_req = p_name in req_set
    p_type = _resolve_json_type(p_spec)
    p_desc = (p_spec.get("description") or p_spec.get("title") or "").strip()
    init_val = _initial_value_from_schema(p_spec)
    w_layout, w_style = widgets.Layout(width="340px"), {"description_width": "0px"}
    if "enum" in p_spec and p_spec["enum"]:
      inp = widgets.Dropdown(options=p_spec["enum"], value=init_val, layout=w_layout, style=w_style)
    elif p_type == "boolean":
      inp = widgets.Checkbox(value=bool(init_val), description=p_name, indent=False, layout=w_layout)
    elif p_type == "integer":
      inp = widgets.IntText(value=int(init_val) if init_val != "" else 0, layout=w_layout, style=w_style)
    elif p_type == "number":
      inp = widgets.FloatText(value=float(init_val) if init_val != "" else 0.0, layout=w_layout, style=w_style)
    elif p_type in ("array", "object"):
      inp = widgets.Textarea(value=init_val if isinstance(init_val, str) else json.dumps(init_val), placeholder=f"Valid JSON {p_type}", layout=widgets.Layout(width="340px", height="58px"), style=w_style)
    else:
      inp = widgets.Text(value=str(init_val), placeholder=p_desc or f"Enter {p_name}", layout=w_layout, style=w_style)
    _param_widgets[p_name] = (inp, p_spec, is_req)
    badge_cls = "mcp-badge-req" if is_req else "mcp-badge-opt"
    badge_txt = "required" if is_req else "optional"
    label_html = widgets.HTML(
        f"<div class='mcp-ux-root mcp-param-meta' style='width:520px;margin:0;'><div>"
        f"<code style='font-size:13px;font-weight:700;color:#0f172a;margin-right:6px;'>{html.escape(p_name)}</code>"
        f"<span class='mcp-badge-type'>{html.escape(p_type)}</span><span class='{badge_cls}'>{badge_txt}</span></div>"
        f"<div style='font-size:11.5px;color:#64748b;margin-top:2px;'>{html.escape(p_desc)}</div></div>"
    )
    form_rows.append(widgets.HBox([label_html, inp], layout=widgets.Layout(align_items="center", justify_content="space-between", padding="6px 0", border="0 0 1px 0 solid #f1f5f9", width="100%")))
  params_box.children = form_rows

def _collect_arguments() -> Tuple[str, Dict[str, Any]]:
  tool_name = tool_dropdown.value
  if not tool_name:
    raise ValueError("No MCP tool is currently selected.")
  args: Dict[str, Any] = {}
  for p_name, (inp, p_spec, is_req) in _param_widgets.items():
    p_type = _resolve_json_type(p_spec)
    val = inp.value
    if p_type in ("array", "object"):
      raw = str(val).strip()
      if not raw and not is_req:
        continue
      args[p_name] = json.loads(raw)
    elif p_type == "string":
      if val == "" and not is_req:
        continue
      args[p_name] = str(val)
    elif p_type == "integer":
      args[p_name] = int(val)
    elif p_type == "number":
      args[p_name] = float(val)
    elif p_type == "boolean":
      args[p_name] = bool(val)
    else:
      args[p_name] = val
  return tool_name, args

def _on_tool_change(change: Dict[str, Any]) -> None:
  if change.get("name") == "value":
    _rebuild_parameter_form(change.get("new"))

def _sync_session(remint_token: bool) -> None:
  with output_area:
    output_area.clear_output(wait=True)
    display(HTML("<div class='mcp-ux-root'>⏳ Synchronizing MCP session and running <code>tools/list</code>...</div>"))
    try:
      fresh_tools = mcp_client.bootstrap(remint_token=remint_token)
      TOOLS_BY_NAME.clear()
      for t in fresh_tools:
        if t.get("name"):
          TOOLS_BY_NAME[t["name"]] = t
      dashboard_html.value = _render_dashboard_html(mcp_client)
      opts = [(f"🔧 {t['name']}", t["name"]) for t in fresh_tools if t.get("name")]
      tool_dropdown.options = opts
      if opts:
        tool_dropdown.value = opts[0][1]
        _rebuild_parameter_form(tool_dropdown.value)
      output_area.clear_output(wait=True)
      display(HTML(f"<div class='mcp-ux-root' style='color:#047857;font-weight:600;font-size:13px;'>✅ Session synchronized (`mcp-session-id={html.escape(mcp_client.session_id or 'stateless')}`) — discovered {len(fresh_tools)} tool(s).</div>"))
    except Exception as exc:
      output_area.clear_output(wait=True)
      display(HTML(f"<div class='mcp-ux-root' style='color:#dc2626;font-weight:600;'>❌ Handshake error: {html.escape(str(exc))}</div>"))

def _on_execute_clicked(_) -> None:
  with output_area:
    output_area.clear_output(wait=True)
    try:
      target_tool, tool_args = _collect_arguments()
    except Exception as exc:
      display(HTML(f"<div class='mcp-ux-root' style='color:#dc2626;font-weight:600;'>❌ Parameter validation error: {html.escape(str(exc))}</div>"))
      return
    display(HTML(f"<div class='mcp-ux-root' style='color:#1d4ed8;font-weight:600;font-size:13px;'>⏳ Invoking <code>tools/call</code> for <b>{html.escape(target_tool)}</b>...</div>"))
    try:
      call_res = mcp_client.call_tool(target_tool, tool_args)
      dashboard_html.value = _render_dashboard_html(mcp_client)
      output_area.clear_output(wait=True)
      display(HTML(_render_call_result_html(call_res)))
    except Exception as exc:
      output_area.clear_output(wait=True)
      display(HTML(f"<div class='mcp-ux-root' style='color:#dc2626;font-weight:600;'>❌ MCP <code>tools/call</code> request failed: {html.escape(str(exc))}</div>"))

tool_dropdown.observe(_on_tool_change, names="value")
refresh_btn.on_click(lambda _: _sync_session(remint_token=False))
remint_btn.on_click(lambda _: _sync_session(remint_token=True))
call_btn.on_click(_on_execute_clicked)
clear_btn.on_click(lambda _: output_area.clear_output())
_rebuild_parameter_form(tool_dropdown.value)

display(widgets.VBox([
    dashboard_html,
    widgets.HBox([tool_dropdown, refresh_btn, remint_btn], layout=widgets.Layout(align_items="center", gap="8px", margin="2px 0 6px 0")),
    tool_card_html,
    params_box,
    widgets.HBox([call_btn, clear_btn], layout=widgets.Layout(align_items="center", gap="10px", margin="4px 0 8px 0")),
    output_area,
], layout=widgets.Layout(width="100%", max_width="980px")))

## Step 2 — Deploy MCP Proxy for Payments, API Product & Developer App
- **Purpose:** Deploy the Model Context Protocol (MCP) Gateway proxy for the Payments service (`mcp-gateway-payment-product`), register the **AI Payment Package** (`ai-payment-package`) enforcing MCP operation-level allow-listing and quota (`tools/list`, `tools/call/get_account_balance`, and `tools/call/get_recent_transactions` at 30 req/min), and onboard the **Payment Agent App** (`payment-agent@example.com`).
- **AFT Deployments:**
  1. **MCP Payments Proxy:** `aft -i templates/mcp-gateway-payment-product.yaml --organization $APIGEE_ORG --environment $APIGEE_ENVIRONMENT --sa $SERVICE_ACCOUNT`
  2. **API Product / Package:** `aft -i config/products/ai-payment-agent-package.yaml --organization $APIGEE_ORG --environment $APIGEE_ENVIRONMENT --sa $SERVICE_ACCOUNT`
  3. **Developer & App:** `aft -i config/apps/user_apps_mcp_payments.yaml --organization $APIGEE_ORG --environment $APIGEE_ENVIRONMENT --sa $SERVICE_ACCOUNT`
- **Dependencies:** Step 0 (Preflight)

In [ ]:
#@title 2.1 Deploy MCP Proxy for Payments, API Product & Developer App via AFT
# 1) Deploys templates/mcp-gateway-payment-product.yaml via AFT and waits until READY
# 2) Deploys config/products/ai-payment-agent-package.yaml via AFT
# 3) Deploys config/apps/user_apps_mcp_payments.yaml via AFT

os.environ.pop("AFT_REPOSITORY", None)
APIGEE_ENVIRONMENT = APIGEE_ENV
os.environ["APIGEE_ENVIRONMENT"] = APIGEE_ENVIRONMENT

TEMPLATE_PAYMENT_REL_PATH = "templates/mcp-gateway-payment-product.yaml"
FEATURE_PAYMENT_ENDPOINT_REL_PATH = "features/mcp-payment-endpoint.yaml"
FEATURE_APIKEY_VERIFY_REL_PATH = "features/auth-apikey-verify.yaml"
PRODUCT_PAYMENT_REL_PATH = "config/products/ai-payment-agent-package.yaml"
APPS_PAYMENT_REL_PATH = "config/apps/user_apps_mcp_payments.yaml"

required_payment_rel_paths = [
    TEMPLATE_PAYMENT_REL_PATH,
    FEATURE_PAYMENT_ENDPOINT_REL_PATH,
    FEATURE_APIKEY_VERIFY_REL_PATH,
    PRODUCT_PAYMENT_REL_PATH,
    APPS_PAYMENT_REL_PATH,
]
if any(not (REPO_DIR / p).is_file() for p in required_payment_rel_paths):
    print(f"🔄 Fetching latest files from origin/{REPO_BRANCH}...")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], capture_output=True, text=True)
    for rel_p in required_payment_rel_paths:
        if not (REPO_DIR / rel_p).is_file():
            subprocess.run(["git", "-C", str(REPO_DIR), "checkout", f"origin/{REPO_BRANCH}", "--", rel_p], capture_output=True, text=True)

# Ensure features/mcp-payment-endpoint.yaml exists with valid MCP ParsePayload & Cloud Run ID Token config
feat_payment_path = REPO_DIR / FEATURE_PAYMENT_ENDPOINT_REL_PATH
if not feat_payment_path.is_file() or "type: template" in feat_payment_path.read_text(encoding="utf-8"):
    feat_payment_path.parent.mkdir(parents=True, exist_ok=True)
    feat_payment_path.write_text(
        """name: mcp-proxy-payment
displayName: mcp-proxy-payment
type: feature
description: mcp-proxy-paymentservice
documentation: ""
gateway: apigee
schemaVersion: 1.0.0
categories: []
parameters: []
endpoints:
  - name: default
    basePath: /paymentservice/mcp
    routes:
      - name: default
        target: default
    flows:
      - name: PreFlow
        mode: Request
        steps:
          - name: PP-ParsePayload
    faultRules: []
targets:
  - name: default
    url: https://mcp-payments-bbva-1052672297239.europe-west1.run.app/mcp
    flows: []
    faultRules: []
    httpTargetConnection:
      Properties:
        Property:
          - metadata:
              name: response.streaming.enabled
            _text: "true"
          - metadata:
              name: request.streaming.enabled
            _text: "true"
      Authentication:
        GoogleIDToken:
          Audience: https://mcp-payments-bbva-1052672297239.europe-west1.run.app
      URL: https://mcp-payments-bbva-1052672297239.europe-west1.run.app/mcp
policies:
  - name: PP-ParsePayload
    type: ParsePayload
    content:
      ParsePayload:
        metadata:
          async: "false"
          continueOnError: "false"
          enabled: "true"
          name: PP-ParsePayload
        DisplayName: PP-ParsePayload
        Source: request
        PayloadType: JSON-RPC-2.0
        Protocol: MCP
resources: []
tests: []
""",
        encoding="utf-8",
    )

for rel_p in required_payment_rel_paths:
    if not (REPO_DIR / rel_p).is_file():
        raise SystemExit(f"❌ HALT: Required file not found at '{REPO_DIR / rel_p}'. Ensure it is committed to the repo or re-run Step 0.")

def run_aft_payment_deploy(rel_path: str, label: str) -> None:
    cmd = [
        aft_bin,
        "-i", rel_path,
        "--organization", APIGEE_ORG,
        "--environment", APIGEE_ENVIRONMENT,
        "--sa", SERVICE_ACCOUNT,
    ]
    print(f"\n▶ {label}: {' '.join(cmd)}")
    res = subprocess.run(cmd, cwd=str(REPO_DIR), capture_output=True, text=True)
    if res.returncode != 0:
        err_out = (res.stderr or res.stdout).strip()
        raise SystemExit(f"❌ HALT: AFT deployment failed for '{rel_path}' (exit {res.returncode}):\n{err_out}")
    if res.stdout.strip():
        print(res.stdout.strip())

# 1. Deploy the MCP Payments Proxy
PROXY_NAME_MCP_PAYMENT = "mcp-gateway-payment-product"
run_aft_payment_deploy(TEMPLATE_PAYMENT_REL_PATH, "[1/3] Deploying MCP Payments Proxy")
deployed_payment_rev = wait_for_proxy_ready(PROXY_NAME_MCP_PAYMENT, APIGEE_ENVIRONMENT)

# 2. Deploy the API Product / Package (ai-payment-package via ai-payment-agent-package.yaml)
prod_payment_path = REPO_DIR / PRODUCT_PAYMENT_REL_PATH
prod_payment_text = prod_payment_path.read_text(encoding="utf-8")
if not re.search(rf"(?m)^\s*-\s*{re.escape(APIGEE_ENVIRONMENT)}\s*$", prod_payment_text):
    prod_payment_text = re.sub(
        r"(?m)^(\s*environments:\s*\n)",
        rf"\g<1>  - {APIGEE_ENVIRONMENT}\n",
        prod_payment_text,
        count=1,
    )
    prod_payment_path.write_text(prod_payment_text, encoding="utf-8")

run_aft_payment_deploy(PRODUCT_PAYMENT_REL_PATH, "[2/3] Deploying API Product / Package")
print(f"✅ Deployed API Product: {PRODUCT_PAYMENT_REL_PATH}")

# 3. Deploy the Developer and App (user_apps_mcp_payments.yaml)
run_aft_payment_deploy(APPS_PAYMENT_REL_PATH, "[3/3] Deploying Developer & App")
print(f"✅ Deployed Developer & App: {APPS_PAYMENT_REL_PATH}")

MCP_PAYMENT_ENDPOINT = f"https://{HOSTNAME}/paymentservice/mcp"
os.environ.update({
    "PROXY_NAME_MCP_PAYMENT": PROXY_NAME_MCP_PAYMENT,
    "MCP_PAYMENT_ENDPOINT": MCP_PAYMENT_ENDPOINT,
})

print("=" * 72)
print(f"✅ Deployed MCP Proxy    : {PROXY_NAME_MCP_PAYMENT} (rev={deployed_payment_rev})")
print(f"✅ API Product           : ai-payment-package ({PRODUCT_PAYMENT_REL_PATH})")
print(f"✅ Developer App         : Payment Agent App (payment-agent@example.com)")
print(f"✅ MCP Payments Endpoint : {MCP_PAYMENT_ENDPOINT}")
print("=" * 72)

In [ ]:
#@title 2.2 Test MCP Payments Proxy (Standard MCP Protocol & Dynamic Tool UX)
#@markdown ### App Credentials (`Payment Agent App` — supports `x-api-key` & OAuth 2.0)
API_KEY = "payment-agent-app-key-123" #@param {type:"string"}
CLIENT_SECRET = "payment-agent-app-secret-123" #@param {type:"string"}

from dataclasses import dataclass
import html, json, time
from typing import Any, Dict, List, Optional, Tuple
from IPython.display import display, HTML
import ipywidgets as widgets
import requests, urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

_pay_host = str(globals().get("HOSTNAME", "")).strip()
if not _pay_host:
  raise RuntimeError("❌ HOSTNAME is empty. Please run Step 0 first.")

_pay_org = str(globals().get("APIGEE_ORG", "")).strip()
_pay_env = str(globals().get("APIGEE_ENV", "")).strip()
PAYMENT_PROXY_NAME = "mcp-gateway-payment-product"
PAYMENT_ACTIVE_REVISION = wait_for_proxy_ready(PAYMENT_PROXY_NAME, _pay_env) if "wait_for_proxy_ready" in globals() else "latest"
PAYMENT_OAUTH_URL = f"https://{_pay_host}/v1/oauth/token"
PAYMENT_MCP_URL = f"https://{_pay_host}/paymentservice/mcp"

@dataclass
class PaymentStepMetric:
  name: str
  status_code: int
  latency_ms: int
  detail: str
  ok: bool = True

@dataclass
class PaymentMCPCallResult:
  tool_name: str
  request_headers: Dict[str, str]
  request_payload: Dict[str, Any]
  status_code: int
  response_headers: Dict[str, str]
  response_body: Dict[str, Any]
  latency_ms: int
  session_id: Optional[str]
  reinitialized: bool = False

class StandardMCPPaymentsClient:
  """Spec-compliant MCP Streamable HTTP client supporting x-api-key and OAuth 2.0."""
  PROTOCOL_VERSION = "2024-11-05"

  def __init__(self, oauth_url: str, mcp_url: str, api_key: str, client_secret: str):
    self.oauth_url = oauth_url
    self.mcp_url = mcp_url
    self.api_key = api_key.strip()
    self.client_secret = client_secret.strip()
    self.auth_mode: str = "x-api-key"
    self.access_token: Optional[str] = None
    self.token_expires_in: Optional[Any] = None
    self.session_id: Optional[str] = None
    self.protocol_version: str = self.PROTOCOL_VERSION
    self.server_info: Dict[str, Any] = {}
    self.server_capabilities: Dict[str, Any] = {}
    self.tools: List[Dict[str, Any]] = []
    self.handshake_metrics: List[PaymentStepMetric] = []
    self._rpc_id = 0
    self._http = requests.Session()
    self._http.verify = False

  def _next_id(self) -> int:
    self._rpc_id += 1
    return self._rpc_id

  @property
  def masked_credential(self) -> str:
    cred = self.access_token if self.auth_mode == "oauth2" and self.access_token else self.api_key
    if not cred:
      return "none"
    return "***" if len(cred) <= 12 else f"{cred[:8]}...{cred[-4:]}"

  def _mcp_headers(self, include_session: bool = True, mask_auth: bool = False) -> Dict[str, str]:
    hdrs: Dict[str, str] = {
        "Content-Type": "application/json",
        "Accept": "application/json, text/event-stream",
    }
    if self.auth_mode == "oauth2" and self.access_token:
      tok = self.masked_credential if mask_auth else self.access_token
      hdrs["Authorization"] = f"Bearer {tok}"
      hdrs["x-api-key"] = self.masked_credential if mask_auth else self.api_key
    else:
      hdrs["x-api-key"] = self.masked_credential if mask_auth else self.api_key
    if include_session and self.session_id:
      hdrs["mcp-session-id"] = self.session_id
      hdrs["mcp-protocol-version"] = self.protocol_version
    return hdrs

  @staticmethod
  def _parse_response(resp: requests.Response) -> Dict[str, Any]:
    ctype = (resp.headers.get("Content-Type") or "").lower()
    raw = (resp.text or "").strip()
    if not raw:
      return {}
    if "text/event-stream" in ctype or raw.startswith(("event:", "data:")):
      last_event: Dict[str, Any] = {}
      for line in raw.splitlines():
        s = line.strip()
        if s.startswith("data:") and s[5:].strip():
          try:
            last_event = json.loads(s[5:].strip())
          except ValueError:
            pass
      if last_event:
        return last_event
    try:
      return resp.json()
    except ValueError:
      return {"raw_text": raw}

  def _capture_session_id(self, resp: requests.Response) -> None:
    for k, v in resp.headers.items():
      if k.lower() == "mcp-session-id" and v:
        self.session_id = v.strip()
        return

  def mint_oauth_token(self) -> PaymentStepMetric:
    t0 = time.perf_counter()
    resp = self._http.post(
        self.oauth_url,
        auth=(self.api_key, self.client_secret),
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        data={"grant_type": "client_credentials"},
        timeout=20,
    )
    ms = int((time.perf_counter() - t0) * 1000)
    if resp.status_code != 200:
      raise RuntimeError(f"OAuth 2.0 token request failed (HTTP {resp.status_code}, {ms}ms): {resp.text[:400]}")
    data = resp.json()
    self.access_token = data.get("access_token", "")
    self.token_expires_in = data.get("expires_in", "N/A")
    self.auth_mode = "oauth2"
    return PaymentStepMetric("1. OAuth 2.0 Token", resp.status_code, ms, f"token={self.masked_credential} (TTL {self.token_expires_in}s)")

  def initialize_session(self, force_oauth: bool = False) -> List[PaymentStepMetric]:
    metrics: List[PaymentStepMetric] = []
    if force_oauth:
      metrics.append(self.mint_oauth_token())
    else:
      self.auth_mode = "x-api-key"
    self.session_id = None
    init_payload = {
        "jsonrpc": "2.0",
        "id": self._next_id(),
        "method": "initialize",
        "params": {
            "protocolVersion": self.PROTOCOL_VERSION,
            "capabilities": {},
            "clientInfo": {"name": "apigee-mcp-payments-colab", "version": "1.0.0"},
        },
    }
    t0 = time.perf_counter()
    resp = self._http.post(self.mcp_url, headers=self._mcp_headers(include_session=False), json=init_payload, timeout=30)
    init_ms = int((time.perf_counter() - t0) * 1000)
    if resp.status_code == 401 and self.auth_mode == "x-api-key" and self.client_secret:
      metrics.append(self.mint_oauth_token())
      t0 = time.perf_counter()
      resp = self._http.post(self.mcp_url, headers=self._mcp_headers(include_session=False), json=init_payload, timeout=30)
      init_ms = int((time.perf_counter() - t0) * 1000)
    elif self.auth_mode == "x-api-key":
      metrics.append(PaymentStepMetric("1. API Key Auth", resp.status_code, init_ms, f"x-api-key={self.masked_credential} (VA-VerifyKey)"))
    self._capture_session_id(resp)
    body = self._parse_response(resp)
    if resp.status_code != 200 or "error" in body:
      raise RuntimeError(f"MCP initialize failed (HTTP {resp.status_code}, {init_ms}ms): {json.dumps(body, ensure_ascii=False)}")
    result = body.get("result", {})
    self.protocol_version = result.get("protocolVersion", self.PROTOCOL_VERSION)
    self.server_info = result.get("serverInfo", {})
    self.server_capabilities = result.get("capabilities", {})
    srv_name = self.server_info.get("name", "mcp-payments-server")
    srv_ver = self.server_info.get("version", "")
    metrics.append(PaymentStepMetric("2. MCP initialize", resp.status_code, init_ms, f"{srv_name} v{srv_ver} • proto {self.protocol_version}"))

    notif_payload = {"jsonrpc": "2.0", "method": "notifications/initialized"}
    t1 = time.perf_counter()
    notif_resp = self._http.post(self.mcp_url, headers=self._mcp_headers(include_session=True), json=notif_payload, timeout=20)
    notif_ms = int((time.perf_counter() - t1) * 1000)
    self._capture_session_id(notif_resp)
    metrics.append(PaymentStepMetric("3. notifications/initialized", notif_resp.status_code, notif_ms, f"session={self.session_id or 'stateless'}"))
    return metrics

  def list_tools(self) -> PaymentStepMetric:
    if not self.session_id and not self.server_info:
      self.initialize_session()
    payload = {"jsonrpc": "2.0", "id": self._next_id(), "method": "tools/list"}
    t0 = time.perf_counter()
    resp = self._http.post(self.mcp_url, headers=self._mcp_headers(include_session=True), json=payload, timeout=30)
    ms = int((time.perf_counter() - t0) * 1000)
    self._capture_session_id(resp)
    body = self._parse_response(resp)
    if resp.status_code != 200 or "error" in body:
      raise RuntimeError(f"MCP tools/list failed (HTTP {resp.status_code}, {ms}ms): {json.dumps(body, ensure_ascii=False)}")
    self.tools = body.get("result", {}).get("tools", [])
    return PaymentStepMetric("4. MCP tools/list", resp.status_code, ms, f"{len(self.tools)} tool(s) discovered")

  def bootstrap(self, force_oauth: bool = False) -> List[Dict[str, Any]]:
    self.handshake_metrics.clear()
    init_metrics = self.initialize_session(force_oauth=force_oauth)
    m_list = self.list_tools()
    self.handshake_metrics.extend(init_metrics)
    self.handshake_metrics.append(m_list)
    return self.tools

  def call_tool(self, name: str, arguments: Dict[str, Any]) -> PaymentMCPCallResult:
    if not isinstance(arguments, dict):
      raise ValueError(f"MCP tools/call 'arguments' must be a dict, got {type(arguments).__name__}")
    payload = {"jsonrpc": "2.0", "id": self._next_id(), "method": "tools/call", "params": {"name": name, "arguments": arguments}}
    t0 = time.perf_counter()
    resp = self._http.post(self.mcp_url, headers=self._mcp_headers(include_session=True), json=payload, timeout=30)
    ms = int((time.perf_counter() - t0) * 1000)
    body = self._parse_response(resp)
    reinitialized = False
    err_msg = str(body.get("error", {}).get("message", "")).lower() if isinstance(body, dict) else ""
    if resp.status_code == 404 or "before initialization" in err_msg or "session" in err_msg:
      self.initialize_session(force_oauth=(self.auth_mode == "oauth2"))
      reinitialized = True
      t0 = time.perf_counter()
      resp = self._http.post(self.mcp_url, headers=self._mcp_headers(include_session=True), json=payload, timeout=30)
      ms = int((time.perf_counter() - t0) * 1000)
      body = self._parse_response(resp)
    interesting_resp_hdrs = {k: v for k, v in resp.headers.items() if k.lower() in ("content-type", "mcp-session-id", "date", "server", "via")}
    return PaymentMCPCallResult(
        tool_name=name,
        request_headers=self._mcp_headers(include_session=True, mask_auth=True),
        request_payload=payload,
        status_code=resp.status_code,
        response_headers=interesting_resp_hdrs,
        response_body=body,
        latency_ms=ms,
        session_id=self.session_id,
        reinitialized=reinitialized,
    )

MCP_PAY_UX_CSS = """
<style>
.mcp-pay-root { font-family: 'Google Sans', 'Roboto', -apple-system, sans-serif; color: #1f2937; max-width: 980px; margin: 6px 0 12px 0; }
.mcp-pay-hero { background: linear-gradient(135deg, #0f172a 0%, #065f46 100%); color: #f8fafc; border-radius: 12px; padding: 16px 20px; box-shadow: 0 4px 14px rgba(15,23,42,0.14); margin-bottom: 12px; }
.mcp-pay-hero-top { display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap; gap: 8px; margin-bottom: 10px; }
.mcp-pay-hero-title { font-size: 16px; font-weight: 700; letter-spacing: 0.2px; display: flex; align-items: center; gap: 8px; }
.mcp-pay-pill { display: inline-flex; align-items: center; gap: 5px; font-size: 11px; font-weight: 600; padding: 3px 10px; border-radius: 999px; background: rgba(255,255,255,0.14); color: #e2e8f0; border: 1px solid rgba(255,255,255,0.2); }
.mcp-pay-pill-green { background: rgba(16,185,129,0.25); color: #6ee7b7; border-color: rgba(16,185,129,0.45); }
.mcp-pay-meta-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(210px, 1fr)); gap: 8px; font-size: 12px; background: rgba(15,23,42,0.45); padding: 10px 12px; border-radius: 8px; border: 1px solid rgba(255,255,255,0.1); }
.mcp-pay-meta-item span { color: #94a3b8; display: block; font-size: 10px; text-transform: uppercase; letter-spacing: 0.5px; }
.mcp-pay-meta-item code { color: #f1f5f9; font-family: 'JetBrains Mono', monospace; font-size: 11.5px; word-break: break-all; }
.mcp-pay-steps-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(205px, 1fr)); gap: 8px; margin-top: 10px; }
.mcp-pay-step-card { background: rgba(255,255,255,0.08); border: 1px solid rgba(255,255,255,0.15); border-radius: 8px; padding: 8px 10px; }
.mcp-pay-step-hdr { display: flex; justify-content: space-between; align-items: center; font-size: 11.5px; font-weight: 600; color: #f8fafc; margin-bottom: 3px; }
.mcp-pay-step-ms { font-size: 10.5px; color: #34d399; font-family: monospace; }
.mcp-pay-step-sub { font-size: 11px; color: #cbd5e1; font-family: monospace; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }
.mcp-pay-tool-card { background: #ffffff; border: 1px solid #e2e8f0; border-left: 4px solid #059669; border-radius: 10px; padding: 12px 16px; margin: 6px 0; box-shadow: 0 1px 3px rgba(0,0,0,0.04); }
.mcp-pay-tool-hdr { display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap; gap: 8px; margin-bottom: 6px; }
.mcp-pay-tool-name { font-family: 'JetBrains Mono', monospace; font-size: 14px; font-weight: 700; color: #065f46; background: #ecfdf5; padding: 3px 10px; border-radius: 6px; border: 1px solid #a7f3d0; }
.mcp-pay-tool-desc { font-size: 13px; color: #334155; line-height: 1.45; margin: 4px 0 8px 0; }
.mcp-pay-schema-details summary { font-size: 11.5px; color: #475569; cursor: pointer; user-select: none; font-weight: 600; }
.mcp-pay-schema-pre { background: #0f172a; color: #e2e8f0; padding: 10px 12px; border-radius: 8px; font-size: 11.5px; font-family: 'JetBrains Mono', monospace; overflow-x: auto; margin-top: 6px; }
.mcp-pay-badge-req { background: #fef2f2; color: #dc2626; border: 1px solid #fecaca; font-size: 10px; font-weight: 700; padding: 1px 6px; border-radius: 4px; text-transform: uppercase; margin-right: 5px; }
.mcp-pay-badge-opt { background: #f1f5f9; color: #64748b; border: 1px solid #cbd5e1; font-size: 10px; font-weight: 600; padding: 1px 6px; border-radius: 4px; text-transform: uppercase; margin-right: 5px; }
.mcp-pay-badge-type { background: #ecfdf5; color: #047857; border: 1px solid #a7f3d0; font-size: 10px; font-family: monospace; padding: 1px 6px; border-radius: 4px; margin-right: 6px; }
.mcp-pay-res-card { border-radius: 10px; border: 1px solid #e2e8f0; background: #ffffff; overflow: hidden; box-shadow: 0 2px 8px rgba(15,23,42,0.06); margin-top: 4px; }
.mcp-pay-res-banner { padding: 10px 16px; display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap; gap: 8px; font-weight: 700; font-size: 13px; }
.mcp-pay-res-ok { background: #ecfdf5; color: #065f46; border-bottom: 1px solid #a7f3d0; }
.mcp-pay-res-err { background: #fef2f2; color: #991b1b; border-bottom: 1px solid #fecaca; }
.mcp-pay-res-warn { background: #fffbeb; color: #92400e; border-bottom: 1px solid #fde68a; }
.mcp-pay-res-body { padding: 14px 16px; }
.mcp-pay-output-box { background: #f8fafc; border: 1px solid #cbd5e1; border-left: 4px solid #10b981; border-radius: 8px; padding: 12px 14px; margin-bottom: 12px; }
.mcp-pay-output-title { font-size: 11.5px; font-weight: 700; text-transform: uppercase; letter-spacing: 0.4px; color: #047857; margin-bottom: 6px; }
.mcp-pay-kv-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(170px, 1fr)); gap: 8px; margin-bottom: 8px; }
.mcp-pay-kv-card { background: #ffffff; border: 1px solid #e2e8f0; border-radius: 6px; padding: 8px 10px; }
.mcp-pay-kv-key { font-size: 10.5px; color: #64748b; text-transform: uppercase; letter-spacing: 0.3px; }
.mcp-pay-kv-val { font-size: 14px; font-weight: 700; color: #0f172a; font-family: 'JetBrains Mono', monospace; margin-top: 2px; }
.mcp-pay-wire-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(360px, 1fr)); gap: 12px; }
.mcp-pay-wire-panel { background: #0f172a; color: #e2e8f0; border-radius: 8px; padding: 10px 12px; font-family: 'JetBrains Mono', monospace; font-size: 11.5px; overflow-x: auto; }
.mcp-pay-wire-hdr { color: #94a3b8; font-size: 11px; font-weight: 700; text-transform: uppercase; letter-spacing: 0.5px; border-bottom: 1px solid #1e293b; padding-bottom: 6px; margin-bottom: 8px; display: flex; justify-content: space-between; }
.mcp-pay-wire-pre { margin: 0; white-space: pre-wrap; word-break: break-word; color: #f1f5f9; }
</style>
"""

def _render_payment_dashboard_html(client: StandardMCPPaymentsClient) -> str:
  steps_html = "".join(
      f"<div class='mcp-pay-step-card'><div class='mcp-pay-step-hdr'><span>✅ {html.escape(m.name)}</span>"
      f"<span class='mcp-pay-step-ms'>HTTP {m.status_code} • {m.latency_ms}ms</span></div>"
      f"<div class='mcp-pay-step-sub' title='{html.escape(m.detail)}'>{html.escape(m.detail)}</div></div>"
      for m in client.handshake_metrics
  )
  srv_title = client.server_info.get("name", "MCP Payments Server")
  srv_ver = client.server_info.get("version", "")
  auth_badge = "OAuth 2.0 Bearer" if client.auth_mode == "oauth2" else "API Key (x-api-key)"
  return (
      f"{MCP_PAY_UX_CSS}<div class='mcp-pay-root'><div class='mcp-pay-hero'>"
      f"<div class='mcp-pay-hero-top'><div class='mcp-pay-hero-title'>💳 Apigee MCP Payments Gateway — Live Session & Tool Explorer</div>"
      f"<div><span class='mcp-pay-pill mcp-pay-pill-green'>● SESSION READY</span> "
      f"<span class='mcp-pay-pill'>Auth: {html.escape(auth_badge)}</span> "
      f"<span class='mcp-pay-pill'>Server: {html.escape(srv_title)} {html.escape(srv_ver)}</span> "
      f"<span class='mcp-pay-pill'>MCP {html.escape(client.protocol_version)}</span> "
      f"<span class='mcp-pay-pill'>Proxy: {html.escape(PAYMENT_PROXY_NAME)} (rev {html.escape(str(PAYMENT_ACTIVE_REVISION))})</span></div></div>"
      f"<div class='mcp-pay-meta-grid'>"
      f"<div class='mcp-pay-meta-item'><span>MCP Payments Endpoint</span><code>{html.escape(client.mcp_url)}</code></div>"
      f"<div class='mcp-pay-meta-item'><span>API Product / Package</span><code>ai-payment-package</code></div>"
      f"<div class='mcp-pay-meta-item'><span>App Key &amp; Credential</span><code>{html.escape(client.api_key)} ({html.escape(client.masked_credential)})</code></div>"
      f"<div class='mcp-pay-meta-item'><span>Active MCP Session ID</span><code>{html.escape(client.session_id or 'stateless')}</code></div></div>"
      f"<div class='mcp-pay-steps-grid'>{steps_html}</div></div></div>"
  )

def _render_payment_tool_card_html(tool_def: Dict[str, Any]) -> str:
  t_name = tool_def.get("name", "unnamed")
  t_desc = (tool_def.get("description") or "No description provided by MCP server.").strip()
  schema = tool_def.get("inputSchema") or {}
  props = schema.get("properties") or {}
  req_set = set(schema.get("required") or [])
  schema_json = json.dumps(schema, indent=2, ensure_ascii=False)
  return (
      f"<div class='mcp-pay-root'><div class='mcp-pay-tool-card'><div class='mcp-pay-tool-hdr'>"
      f"<div><span class='mcp-pay-tool-name'>🔧 {html.escape(t_name)}</span></div>"
      f"<div style='font-size:12px;color:#475569;font-weight:600;'>{len(props)} parameter(s) • {len(req_set)} required</div></div>"
      f"<div class='mcp-pay-tool-desc'>{html.escape(t_desc)}</div>"
      f"<details class='mcp-pay-schema-details'><summary>📐 Inspect Discovered <code>inputSchema</code> from <code>tools/list</code></summary>"
      f"<pre class='mcp-pay-schema-pre'>{html.escape(schema_json)}</pre></details></div></div>"
  )

def _render_payment_call_result_html(res: PaymentMCPCallResult) -> str:
  body = res.response_body
  has_rpc_error = isinstance(body, dict) and "error" in body
  result_obj = body.get("result", {}) if isinstance(body, dict) else {}
  is_tool_error = isinstance(result_obj, dict) and bool(result_obj.get("isError"))
  if res.status_code == 200 and not has_rpc_error and not is_tool_error:
    banner_cls, status_label = "mcp-pay-res-ok", f"✅ ALLOWED BY APIGEE MCP GATEWAY • HTTP {res.status_code} OK"
  elif res.status_code in (401, 403, 429):
    banner_cls, status_label = "mcp-pay-res-err", f"🛡️ BLOCKED BY APIGEE MCP GATEWAY • HTTP {res.status_code}"
  else:
    banner_cls, status_label = "mcp-pay-res-warn", f"⚠️ MCP SERVER / JSON-RPC ERROR • HTTP {res.status_code}"
  reinit_badge = " • 🔄 Session Auto-Reinitialized" if res.reinitialized else ""
  meta_right = f"tool={html.escape(res.tool_name)} • latency={res.latency_ms}ms • session={html.escape(res.session_id or 'stateless')}{reinit_badge}"

  content_blocks_html = ""
  if res.status_code == 200 and not has_rpc_error:
    content_items = result_obj.get("content", []) if isinstance(result_obj, dict) else []
    rendered_items = []
    for item in content_items:
      if isinstance(item, dict) and item.get("type") == "text":
        txt = item.get("text", "")
        try:
          parsed_json = json.loads(txt)
          if isinstance(parsed_json, dict):
            kv_cards = "".join(
                f"<div class='mcp-pay-kv-card'><div class='mcp-pay-kv-key'>{html.escape(str(k))}</div>"
                f"<div class='mcp-pay-kv-val'>{html.escape(str(v))}</div></div>"
                for k, v in parsed_json.items() if not isinstance(v, (dict, list))
            )
            kv_sec = f"<div class='mcp-pay-kv-grid'>{kv_cards}</div>" if kv_cards else ""
            rendered_items.append(f"{kv_sec}<pre style='margin:0;font-family:monospace;font-size:12px;color:#1e293b;'>{html.escape(json.dumps(parsed_json, indent=2, ensure_ascii=False))}</pre>")
          else:
            rendered_items.append(f"<pre style='margin:0;font-family:monospace;font-size:12.5px;color:#0f172a;'>{html.escape(json.dumps(parsed_json, indent=2, ensure_ascii=False))}</pre>")
        except ValueError:
          rendered_items.append(f"<pre style='margin:0;font-family:monospace;font-size:12.5px;color:#0f172a;white-space:pre-wrap;'>{html.escape(txt)}</pre>")
      else:
        rendered_items.append(f"<pre style='margin:0;font-family:monospace;font-size:12px;'>{html.escape(json.dumps(item, indent=2, ensure_ascii=False))}</pre>")
    if rendered_items:
      box_border = "#f59e0b" if is_tool_error else "#10b981"
      title_str = "⚠️ Tool Execution Error Output (result.content)" if is_tool_error else "🎯 MCP Tool Output (result.content)"
      content_blocks_html = f"<div class='mcp-pay-output-box' style='border-left-color:{box_border};'><div class='mcp-pay-output-title'>{title_str}</div>{''.join(rendered_items)}</div>"
  elif has_rpc_error:
    err_pretty = json.dumps(body["error"], indent=2, ensure_ascii=False)
    content_blocks_html = f"<div class='mcp-pay-output-box' style='border-left-color:#ef4444;background:#fef2f2;'><div class='mcp-pay-output-title' style='color:#991b1b;'>⚠️ JSON-RPC 2.0 Error Object</div><pre style='margin:0;font-family:monospace;font-size:12px;color:#7f1d1d;'>{html.escape(err_pretty)}</pre></div>"

  req_wire = json.dumps({"headers": res.request_headers, "body": res.request_payload}, indent=2, ensure_ascii=False)
  resp_wire = json.dumps({"status": res.status_code, "headers": res.response_headers, "body": res.response_body}, indent=2, ensure_ascii=False)
  return (
      f"<div class='mcp-pay-root'><div class='mcp-pay-res-card'><div class='mcp-pay-res-banner {banner_cls}'>"
      f"<span>{status_label}</span><span style='font-family:monospace;font-size:11.5px;'>{meta_right}</span></div>"
      f"<div class='mcp-pay-res-body'>{content_blocks_html}<div class='mcp-pay-wire-grid'>"
      f"<div class='mcp-pay-wire-panel'><div class='mcp-pay-wire-hdr'><span>📤 MCP JSON-RPC 2.0 Request</span><span>POST /paymentservice/mcp</span></div><pre class='mcp-pay-wire-pre'>{html.escape(req_wire)}</pre></div>"
      f"<div class='mcp-pay-wire-panel'><div class='mcp-pay-wire-hdr'><span>📥 MCP JSON-RPC 2.0 Response</span><span>HTTP {res.status_code} ({res.latency_ms}ms)</span></div><pre class='mcp-pay-wire-pre'>{html.escape(resp_wire)}</pre></div>"
      f"</div></div></div></div>"
  )

mcp_pay_client = StandardMCPPaymentsClient(PAYMENT_OAUTH_URL, PAYMENT_MCP_URL, API_KEY, CLIENT_SECRET)
pay_discovered_tools = mcp_pay_client.bootstrap(force_oauth=False)
PAY_TOOLS_BY_NAME: Dict[str, Dict[str, Any]] = {t["name"]: t for t in pay_discovered_tools if t.get("name")}

pay_dashboard_html = widgets.HTML(value=_render_payment_dashboard_html(mcp_pay_client))
pay_tool_card_html = widgets.HTML()
pay_tool_options = [(f"🔧 {t['name']}", t["name"]) for t in pay_discovered_tools if t.get("name")]
pay_tool_dropdown = widgets.Dropdown(options=pay_tool_options, value=pay_tool_options[0][1] if pay_tool_options else None, description="Select Tool:", style={"description_width": "95px"}, layout=widgets.Layout(width="440px", height="36px"))
pay_refresh_btn = widgets.Button(description="🔄 Re-sync tools/list", button_style="info", tooltip="Re-run MCP initialize + tools/list handshake", layout=widgets.Layout(width="180px", height="36px"))
pay_oauth_btn = widgets.Button(description="🔑 Switch / Mint OAuth", button_style="", tooltip="Mint an OAuth 2.0 Bearer token and re-initialize MCP session", layout=widgets.Layout(width="185px", height="36px"))
pay_params_box = widgets.VBox(layout=widgets.Layout(background="#f8fafc", border="1px solid #e2e8f0", padding="12px 16px", margin="6px 0 10px 0", width="100%", max_width="980px"))
pay_call_btn = widgets.Button(description="▶ Execute tools/call", button_style="success", tooltip="Send JSON-RPC 2.0 tools/call request over active MCP Payments session", layout=widgets.Layout(width="220px", height="40px"))
pay_clear_btn = widgets.Button(description="🧹 Clear Output", button_style="", layout=widgets.Layout(width="140px", height="40px"))
pay_output_area = widgets.Output(layout=widgets.Layout(width="100%", max_width="980px"))
_pay_param_widgets: Dict[str, Tuple[widgets.Widget, Dict[str, Any], bool]] = {}

def _pay_resolve_json_type(prop_spec: Dict[str, Any]) -> str:
  raw_type = prop_spec.get("type", "string")
  return next((t for t in raw_type if t != "null"), "string") if isinstance(raw_type, list) else str(raw_type)

def _pay_initial_value_from_schema(prop_spec: Dict[str, Any]) -> Any:
  if "default" in prop_spec and prop_spec["default"] is not None:
    return prop_spec["default"]
  if "enum" in prop_spec and prop_spec["enum"]:
    return prop_spec["enum"][0]
  p_type = _pay_resolve_json_type(prop_spec)
  return False if p_type == "boolean" else (0 if p_type == "integer" else (0.0 if p_type == "number" else ("[]" if p_type == "array" else ("{}" if p_type == "object" else ""))))

def _rebuild_payment_parameter_form(tool_name: Optional[str]) -> None:
  _pay_param_widgets.clear()
  if not tool_name or tool_name not in PAY_TOOLS_BY_NAME:
    pay_tool_card_html.value = "<div class='mcp-pay-root'><i>No MCP tools available.</i></div>"
    pay_params_box.children = []
    return
  tool_def = PAY_TOOLS_BY_NAME[tool_name]
  pay_tool_card_html.value = _render_payment_tool_card_html(tool_def)
  schema = tool_def.get("inputSchema") or {}
  props = schema.get("properties") or {}
  req_set = set(schema.get("required") or [])
  if not props:
    pay_params_box.children = [widgets.HTML("<div style='font-size:12.5px;color:#475569;'>✨ This tool accepts no arguments (<code>\"arguments\": {}</code>).</div>")]
    return
  form_rows: List[widgets.Widget] = [widgets.HTML("<div style='font-size:12px;font-weight:700;text-transform:uppercase;letter-spacing:0.5px;color:#475569;margin-bottom:6px;'>📝 Dynamic Tool Parameters (Generated from <code>inputSchema</code>)</div>")]
  for p_name, p_spec in props.items():
    is_req = p_name in req_set
    p_type = _pay_resolve_json_type(p_spec)
    p_desc = (p_spec.get("description") or p_spec.get("title") or "").strip()
    init_val = _pay_initial_value_from_schema(p_spec)
    w_layout, w_style = widgets.Layout(width="340px"), {"description_width": "0px"}
    if "enum" in p_spec and p_spec["enum"]:
      inp = widgets.Dropdown(options=p_spec["enum"], value=init_val, layout=w_layout, style=w_style)
    elif p_type == "boolean":
      inp = widgets.Checkbox(value=bool(init_val), description=p_name, indent=False, layout=w_layout)
    elif p_type == "integer":
      inp = widgets.IntText(value=int(init_val) if init_val != "" else 0, layout=w_layout, style=w_style)
    elif p_type == "number":
      inp = widgets.FloatText(value=float(init_val) if init_val != "" else 0.0, layout=w_layout, style=w_style)
    elif p_type in ("array", "object"):
      inp = widgets.Textarea(value=init_val if isinstance(init_val, str) else json.dumps(init_val), placeholder=f"Valid JSON {p_type}", layout=widgets.Layout(width="340px", height="58px"), style=w_style)
    else:
      inp = widgets.Text(value=str(init_val), placeholder=p_desc or f"Enter {p_name}", layout=w_layout, style=w_style)
    _pay_param_widgets[p_name] = (inp, p_spec, is_req)
    badge_cls = "mcp-pay-badge-req" if is_req else "mcp-pay-badge-opt"
    badge_txt = "required" if is_req else "optional"
    label_html = widgets.HTML(
        f"<div class='mcp-pay-root' style='width:520px;margin:0;line-height:1.35;'><div>"
        f"<code style='font-size:13px;font-weight:700;color:#0f172a;margin-right:6px;'>{html.escape(p_name)}</code>"
        f"<span class='mcp-pay-badge-type'>{html.escape(p_type)}</span><span class='{badge_cls}'>{badge_txt}</span></div>"
        f"<div style='font-size:11.5px;color:#64748b;margin-top:2px;'>{html.escape(p_desc)}</div></div>"
    )
    form_rows.append(widgets.HBox([label_html, inp], layout=widgets.Layout(align_items="center", justify_content="space-between", padding="6px 0", border="0 0 1px 0 solid #f1f5f9", width="100%")))
  pay_params_box.children = form_rows

def _collect_payment_arguments() -> Tuple[str, Dict[str, Any]]:
  tool_name = pay_tool_dropdown.value
  if not tool_name:
    raise ValueError("No MCP tool is currently selected.")
  args: Dict[str, Any] = {}
  for p_name, (inp, p_spec, is_req) in _pay_param_widgets.items():
    p_type = _pay_resolve_json_type(p_spec)
    val = inp.value
    if p_type in ("array", "object"):
      raw = str(val).strip()
      if not raw and not is_req:
        continue
      args[p_name] = json.loads(raw)
    elif p_type == "string":
      if val == "" and not is_req:
        continue
      args[p_name] = str(val)
    elif p_type == "integer":
      args[p_name] = int(val)
    elif p_type == "number":
      args[p_name] = float(val)
    elif p_type == "boolean":
      args[p_name] = bool(val)
    else:
      args[p_name] = val
  return tool_name, args

def _on_pay_tool_change(change: Dict[str, Any]) -> None:
  if change.get("name") == "value":
    _rebuild_payment_parameter_form(change.get("new"))

def _sync_payment_session(force_oauth: bool) -> None:
  with pay_output_area:
    pay_output_area.clear_output(wait=True)
    display(HTML("<div class='mcp-pay-root'>⏳ Synchronizing MCP Payments session and running <code>tools/list</code>...</div>"))
    try:
      fresh_tools = mcp_pay_client.bootstrap(force_oauth=force_oauth)
      PAY_TOOLS_BY_NAME.clear()
      for t in fresh_tools:
        if t.get("name"):
          PAY_TOOLS_BY_NAME[t["name"]] = t
      pay_dashboard_html.value = _render_payment_dashboard_html(mcp_pay_client)
      opts = [(f"🔧 {t['name']}", t["name"]) for t in fresh_tools if t.get("name")]
      pay_tool_dropdown.options = opts
      if opts:
        pay_tool_dropdown.value = opts[0][1]
        _rebuild_payment_parameter_form(pay_tool_dropdown.value)
      pay_output_area.clear_output(wait=True)
      display(HTML(f"<div class='mcp-pay-root' style='color:#047857;font-weight:600;font-size:13px;'>✅ Session synchronized (`mcp-session-id={html.escape(mcp_pay_client.session_id or 'stateless')}`) — discovered {len(fresh_tools)} tool(s).</div>"))
    except Exception as exc:
      pay_output_area.clear_output(wait=True)
      display(HTML(f"<div class='mcp-pay-root' style='color:#dc2626;font-weight:600;'>❌ Handshake error: {html.escape(str(exc))}</div>"))

def _on_pay_execute_clicked(_) -> None:
  with pay_output_area:
    pay_output_area.clear_output(wait=True)
    try:
      target_tool, tool_args = _collect_payment_arguments()
    except Exception as exc:
      display(HTML(f"<div class='mcp-pay-root' style='color:#dc2626;font-weight:600;'>❌ Parameter validation error: {html.escape(str(exc))}</div>"))
      return
    display(HTML(f"<div class='mcp-pay-root' style='color:#047857;font-weight:600;font-size:13px;'>⏳ Invoking <code>tools/call</code> for <b>{html.escape(target_tool)}</b>...</div>"))
    try:
      call_res = mcp_pay_client.call_tool(target_tool, tool_args)
      pay_dashboard_html.value = _render_payment_dashboard_html(mcp_pay_client)
      pay_output_area.clear_output(wait=True)
      display(HTML(_render_payment_call_result_html(call_res)))
    except Exception as exc:
      pay_output_area.clear_output(wait=True)
      display(HTML(f"<div class='mcp-pay-root' style='color:#dc2626;font-weight:600;'>❌ MCP <code>tools/call</code> request failed: {html.escape(str(exc))}</div>"))

pay_tool_dropdown.observe(_on_pay_tool_change, names="value")
pay_refresh_btn.on_click(lambda _: _sync_payment_session(force_oauth=False))
pay_oauth_btn.on_click(lambda _: _sync_payment_session(force_oauth=True))
pay_call_btn.on_click(_on_pay_execute_clicked)
pay_clear_btn.on_click(lambda _: pay_output_area.clear_output())
_rebuild_payment_parameter_form(pay_tool_dropdown.value)

display(widgets.VBox([
    pay_dashboard_html,
    widgets.HBox([pay_tool_dropdown, pay_refresh_btn, pay_oauth_btn], layout=widgets.Layout(align_items="center", gap="8px", margin="2px 0 6px 0")),
    pay_tool_card_html,
    pay_params_box,
    widgets.HBox([pay_call_btn, pay_clear_btn], layout=widgets.Layout(align_items="center", gap="10px", margin="4px 0 8px 0")),
    pay_output_area,
], layout=widgets.Layout(width="100%", max_width="980px")))